# Eukaryotic toehold switch — ON/OFF report (single switch + AND)

Built from `and_eu_report_spec.md` in this folder. Measures each candidate's real,
ON/OFF behaviour through `FoldEngine` only (no NUPACK, no MFE-only shortcuts) and
decomposes every state into three independent signals —
**`trigger_bound`** (paired to an external strand), **`stem_closed`** (the intended OFF
hairpin actually closed), **`misfolded`** (paired to neither) — rather than collapsing
them into one "open" bit. That collapse is exactly what let an earlier version of this
project's AND construct read 80-90% "open" with **zero triggers present**, purely from
intramolecular cross-hairpin misfolding (see Traps, bottom of the spec).

Reuses `fuse`/`measure`/`load_ranked_designs`/`motif_self_bind_score` from
`_switch_construct.py` and the exact-only parallel workers + `designed_pairs`/
`pick_fake_trigger(s)`/`and_pair_jobs` from `_joint_state_parallel.py` — nothing here
reimplements either module.

**Fake triggers:** a fake (an off-target trigger used for the OFF checks) is a pool trigger
that shares at most `FAKE_MAX_SEQ_OVERLAP_NT` nt of identical sequence with any real trigger
in the state's construct (`pick_fake_trigger`).

`OPEN_THRESHOLD` (see the parameters cell) only sets the HIGH/LOW cutoff of the green/red
plan colouring of the decomposition tables; it never enters a score.

**Method:** every value in the reports is an exact partition-function quantity; nothing is
sampled. `trigger_bound`, `stem_closed` and `misfolded` are exact ensemble means from
`FoldEngine.base_pair_probabilities`. The joint ON probability `P(duplexes)` is
`FoldEngine.constrained_probability`: the exact probability that ALL designed
footprint-trigger base pairs are present at once (every complementary position of each trigger
against its footprint, from the sequence design). OFF is the worst state's stem opening (stem-loosening: the exact share of stem_up not paired to its own stem_down).

**Strand order of two-trigger states:** every state holding both real
triggers is folded as `fused & trigger_B & trigger_A`, not A-then-B. ViennaRNA's
multi-strand model only allows structures nested along the written strand order, and
the A-then-B order makes footprint A / trigger A and footprint B / trigger B cross (a
pseudoknot), so trigger B could not bind and the joint ON state read ~0. The order is
set once in `both_triggers_extra` (`_joint_state_parallel.py`); see the strand-order
verification cell and `and_eu_report_spec.md`.


## Setup

In [ ]:
# Put the shared _fixtures.py (notebooks/ root) and this folder itself
# (_switch_construct.py / _joint_state_parallel.py) on sys.path, then fx.bootstrap()
# adds <repo>/src so `import engine...` resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_fixtures.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break
for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_joint_state_parallel.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()


## Parameters — every tunable in one place (and_eu_report_spec.md Step 1)

In [ ]:
# ============================================================================
# PARAMETERS. If a later cell needs a new threshold, it belongs HERE, not inline.
# ============================================================================

# ---- data source -----------------------------------------------------------
CSV_PATH = (
    "/Users/zeev/Projects/NucSyn_Public_Platform_0.1.1/nucsyn-platform-work/"
    "output_T2/targets/sele/basic_switch/results_ranked.csv"
)
# NucSyn runner export, topology euk_open_5p_kozak_after_stem, already sorted
# best-first by postgen_global_rank. Printed on both reports.

# Column-name map: a required parameter of load_ranked_designs(), not a module default.
COLUMNS = {
    "switch": "switch_strand",
    "trigger": "trigger_seq",
    "domains": "switch_domain_breakdown_json",
    "rank": "postgen_global_rank",
    "trigger_start": "trigger_start_pos",
    "trigger_end": "trigger_end_pos",
}
# CSV columns carried through into design["metrics"] unchanged (CLAUDE.md sec 5: a value
# already computed upstream is read, not re-derived).
CARRY_METRICS = ("sim_score", "mcc", "switch_toehold_unpaired_prob_mean", "energy_trigger_mfe")

# fuse()'s required parameters (spec Step 2, AND construct): the 5' half keeps everything
# before its OWN kozak starts, the 3' half keeps everything after its OWN prefix ends.
CUT_5P_BEFORE = "kozak"
KEEP_3P_AFTER = "prefix"

# ---- folding -----------------------------------------------------------------
TEMPERATURE_C = 37.0  # folding temperature (deg C), recorded on every energy below

# ---- single-switch screen -----------------------------------------------------
SINGLE_POOL_N = 1000    # ranked designs loaded for the single-switch screen
SINGLE_REPORT_N = 20   # how many reach the Single Switch report

# ---- AND grid -------------------------------------------------------------------
AND_GRID_N = 30   # top-N singles (by single-switch ON - OFF) fed into the N x N grid
AND_REPORT_N = 20 # how many AND pairs reach the AND report

# ---- ratio ------------------------------------------------------------------------
OFF_FLOOR_PCT = 1.0   # ratio = ON% / max(OFF%, OFF_FLOOR_PCT), percentage points. An OFF below
# this value is replaced by it in the ratio (the row is marked * and a note is printed).

# ---- plan colouring --------------------------------------------------------------
OPEN_THRESHOLD = 0.70   # HIGH/LOW cutoff of the green/red plan colouring ONLY:
# HIGH >= OPEN_THRESHOLD, LOW <= 1 - OPEN_THRESHOLD. It never enters a score or a ratio.

# ---- fake-trigger counts -- two separate parameters, same default, different reason -
SINGLE_N_FAKES = 10
# Distinct fake triggers per single-switch candidate, averaged into its OFF fakes term.
AND_N_FAKES = 10
# Distinct fakes drawn PER SIDE for each AND pair (A-side and B-side states, two
# independent rotations). Separate from SINGLE_N_FAKES because AND has two footprints
# screened and reported separately: changing one count must never change the other.

# ---- fake-trigger selection and AND trigger-pair screen -------------------------------
FAKE_MAX_SEQ_OVERLAP_NT = 8  # max identical run (nt) a fake trigger may share with a real
# trigger; also the cap for the AND trigger-vs-trigger screen
WOBBLE_MAX_FRACTION = 0.50   # used by the AND trigger-vs-trigger complementarity screen only
# (max share of a counted complementary run that may be G.U wobble); fakes ignore it

# ---- AND spacer --------------------------------------------------------------------
SPACER_VARIANTS = ("no_spacer", "spacer")  # both are run and reported separately
SPACER_MIN_LENGTH = 15
SPACER_MAX_LENGTH = 20
SPACER_SEED = 1                   # literal seed -- never wall-clock (CLAUDE.md sec 6)
SPACER_CANDIDATE_POOL_SIZE = 150  # pool generate_spacer_candidates() draws from

# ---- optional homopolymer filter ----------------------------------------------------
FILTER_HOMOPOLYMER_RUN = False   # exclude toeholds with a run > MAX_HOMOPOLYMER_RUN
MAX_HOMOPOLYMER_RUN = 5

# ---- execution ------------------------------------------------------------------------
MAX_WORKERS = 6   # process-pool width -- one pool call per BATCH, never one per job

# ---- snapshot --------------------------------------------------------------------------
SNAPSHOT_TO_LOAD = "/Users/zeev/Projects/cernal_software/var/reports/and_eu_report_snapshot_20261007T095710.json"
# Set to a snapshot filename (or an absolute path), e.g.
# "and_eu_report_snapshot_20261005T184344.json" (looked up in var/reports/), to skip every
# heavy cell and restore single_report_rows/and_report_rows from a previous full run -- a
# pure load (the cell right after the "Part 1" heading): nothing is recomputed.
# None means a normal full run: every heavy cell runs and a new snapshot is exported.


In [ ]:
# One FoldEngine for the WHOLE notebook (single-switch part and AND part both reuse
# it) -- one cache, one temperature (CLAUDE.md sec 1/sec 5: FoldEngine is the only
# place a gate family folds, and exactly one instance exists per run).
folder = fx.fold_engine(real=True, temperature=TEMPERATURE_C)

# Publish that one engine to the parallel-worker module, which folds through it
# instead of importing ViennaRNA itself. The pools fork, so every worker inherits
# THIS instance -- must happen before any pool starts.
from _joint_state_parallel import use_fold_engine
use_fold_engine(folder)
print(f"FoldEngine published to the worker module: {folder.versions()}")

from _switch_construct import fuse, load_ranked_designs, measure
from _joint_state_parallel import (
    SingleExactJob,
    and_pair_jobs,
    both_triggers_extra,
    generate_spacer_candidates,
    pick_fake_triggers,
    pick_optimized_spacer,
    run_parallel_and_decomp,
    run_parallel_single_exact,
    trigger_pair_too_similar,
)
from engine.sequences import longest_homopolymer

import datetime, html, json, subprocess, time
from pathlib import Path


## Plotting + report CSS helpers

In [ ]:
# ==================================================================================
# Plotting helpers -- same conventions as toehold_and_eu_test.ipynb (2D structure via
# FoldEngine.layout_coordinates, a real base-pair-probability heatmap, a segmented
# layout bar for the domain diagram). Nothing here re-folds anything: every sequence
# plotted was already folded by measure()/the decomposition jobs earlier in this run,
# so folder.mfe/.base_pair_probabilities calls below are cache hits.
# ==================================================================================
import base64
from io import BytesIO

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

_COLOR_A = "#4C6E8C"   # hairpin A / 5' half / single switch's one hairpin
_COLOR_B = "#B5507E"   # hairpin B / 3' half (AND only)
_DEFAULT_FILL = "#C7D0CA"   # a base outside every known domain span
_TRIGGER_COLOR = "#D1495B"  # real trigger strand on the ON structure plot; also the ring
                             # marking a base paired to a hidden trigger strand
_SPACER_COLOR = "#3F8F8C"   # one teal for both palettes: the spacer belongs to neither hairpin
_PAYLOAD_RING_COLOR = "#B5760E"   # ring on a base paired into the construct's own hidden
                             # payload tail (a real fold-back, not an error); same amber as
                             # the exp_gene domain colour, so it reads as "payload"

# Domain colours, chosen to stay distinct from each other and from _TRIGGER_COLOR /
# _SPACER_COLOR / _DEFAULT_FILL.
_DOMAIN_COLOR_A = {
    "prefix": "#C9A227", "toehold": "#4C6E8C", "stem_up": "#7E97AC",
    "loop": "#8E6AA8", "stem_down": "#7E97AC", "kozak": "#2E9E56", "exp_gene": "#B5760E",
    "spacer": _SPACER_COLOR,
}
# AND needs a doubled palette so hairpin A and B are distinguishable (spec Step 4) -- same
# domain names, a second, B-tinted hue each.
_DOMAIN_COLOR_B = {
    "prefix": "#B98B5E", "toehold": "#B5507E", "stem_up": "#C98CA8",
    "loop": "#6B4E79", "stem_down": "#C98CA8", "kozak": "#2E9E56", "exp_gene": "#8E6A3A",
    "spacer": _SPACER_COLOR,
}

# Domains dropped from the RENDERED layout bar and 2D structure plots. The payload stays a
# real domain in the underlying data; this only decides what gets drawn. Shared by the
# single-switch and AND images cells.
_PAYLOAD_DOMAINS = ("exp_gene", "opt_exp_gene")


def _png_from_fig(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", facecolor="white")
    plt.close(fig)
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("ascii")


def _domain_fill_from_domains(domains, palette, label_suffix=""):
    """Builds the ``domain_fill`` list :func:`structure_png` wants -- (start, end,
    color, label) -- from a plain ``{name: (start, end)}`` domain-breakdown dict and a
    name->color palette. ``label_suffix`` disambiguates two hairpins sharing domain names in the AND report
    (" (A)"/" (B)") so the legend shows both, not one overwriting the other.
    """
    return [
        (s, e, palette.get(name, _DEFAULT_FILL), f"{name}{label_suffix}")
        for name, (s, e) in domains.items()
        if e > s
    ]


def structure_png(sequence, structure, title, spans=(), domain_fill=(), figsize=(3.6, 3.2),
                   visible_len=None, construct_len=None, legend_fontsize=5.6, legend_ncol=4,
                   legend_below_figure=False):
    """One folded structure as a base64 PNG. Coordinates from
    ``FoldEngine.layout_coordinates`` (the shared 2D layout).

    ``domain_fill``: ``(start, end, color, label)`` tuples -- the per-domain fill color
    for every base in that span, with a legend built from the unique ``(label, color)``
    pairs actually present. A base outside every span keeps the flat ``_DEFAULT_FILL``
    (e.g. a trigger strand, which has no domain breakdown of its own).

    ``spans``: ``(start, end, color)`` tuples -- ring highlights drawn on top of the fill
    (e.g. the trigger-bound footprint). Independent of ``domain_fill``: fill says which
    domain a base belongs to, the ring highlights a footprint.

    ``visible_len``: when set, only positions ``0..visible_len`` of ``sequence`` (the
    switch/fused construct's OWN bases) are drawn; any appended trigger strand (or any base
    at/after this index) is never scattered. A base pair with ONE partner outside
    ``visible_len`` -- a switch base paired to a hidden base -- draws no connecting line
    but the visible base gets a ring (a dangling "bound externally" stub). A pair with
    BOTH partners outside is skipped. ``None`` draws everything.

    ``construct_len``: length of the construct part alone (``len(fused)``), always
    ``<= len(sequence)``; distinguishes WHY a dangling partner is hidden, for the ring's
    colour. A hidden partner index ``< construct_len`` is the construct's OWN hidden tail
    (e.g. the clipped payload past kozak, a real fold-back) and gets
    ``_PAYLOAD_RING_COLOR``; ``>= construct_len`` is a hidden TRIGGER strand and gets
    ``_TRIGGER_COLOR``. Defaults to ``visible_len`` (every dangling ring reads as
    "trigger"), correct where ``sequence`` never has a construct-internal hidden tail
    (the single-switch report folds an already-truncated sequence).

    ``legend_fontsize`` / ``legend_ncol``: legend text size and column count.
    ``legend_below_figure``: when True the legend is a FIGURE legend in a reserved strip
    at the bottom (``tight_layout`` rect sized from its row count), so the PNG has a fixed
    ``figsize`` and no label is clipped at the left/right edge. False (default) places the
    legend inside the axes (single-switch report).
    """
    points = folder.layout_coordinates(structure)
    xs = [x for x, _ in points]
    ys = [y for _, y in points]
    _vis = len(sequence) if visible_len is None else visible_len
    _construct_len = _vis if construct_len is None else construct_len

    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    ax.plot(xs[:_vis], ys[:_vis], "-", color="#B9C2BC", linewidth=1.1, zorder=1)
    stack = []
    _cross_bound = {}  # visible base index -> its own hidden partner's index
    for i, ch in enumerate(structure):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            j = stack.pop()
            i_vis, j_vis = i < _vis, j < _vis
            if i_vis and j_vis:
                ax.plot([xs[i], xs[j]], [ys[i], ys[j]], "-", color="#8899AA",
                         linewidth=0.9, zorder=1, alpha=0.7)
            elif i_vis or j_vis:
                _cross_bound[i if i_vis else j] = j if i_vis else i
            # else: neither partner visible -- entirely among hidden bases, skipped.
    _drew_payload_ring, _drew_trigger_ring = False, False
    for i in range(_vis):
        fill = next((c for s, e, c, _l in domain_fill if s <= i < e), _DEFAULT_FILL)
        ring = next((c for s, e, c in spans if s <= i < e), None)
        if ring is None and i in _cross_bound:
            # Dangling-stub indicator: this visible base is paired to a hidden base that
            # is deliberately not drawn. A partner still below construct_len is the
            # construct's OWN hidden tail (a real fold-back), not a trigger.
            if _cross_bound[i] < _construct_len:
                ring = _PAYLOAD_RING_COLOR
                _drew_payload_ring = True
            else:
                ring = _TRIGGER_COLOR
                _drew_trigger_ring = True
        ax.scatter(xs[i], ys[i], s=24 if ring else 13, color=fill,
                    edgecolors=ring or "#666666", linewidths=1.3 if ring else 0.3,
                    zorder=3 if ring else 2)
    ax.text(xs[0], ys[0] + 8, "5'", fontsize=7, ha="center", color="#345")
    ax.text(xs[_vis - 1], ys[_vis - 1] + 8, "3'", fontsize=7, ha="center", color="#345")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")
    if domain_fill or _drew_payload_ring or _drew_trigger_ring:
        seen = []
        for _s, _e, _c, _l in domain_fill:
            if (_l, _c) not in seen:
                seen.append((_l, _c))
        handles = [
            mlines.Line2D([0], [0], marker="o", color="w", markerfacecolor=_color,
                          markeredgecolor="#666666", markersize=6, linewidth=0, label=_label)
            for _label, _color in seen
        ]
        # One legend entry per ring colour actually drawn on this plot -- an open ring,
        # distinct from the filled domain entries above it.
        if _drew_payload_ring:
            handles.append(mlines.Line2D(
                [0], [0], marker="o", color="w", markerfacecolor="none",
                markeredgecolor=_PAYLOAD_RING_COLOR, markeredgewidth=1.3, markersize=7,
                linewidth=0, label="ring: paired into this construct's own hidden tail"))
        if _drew_trigger_ring:
            handles.append(mlines.Line2D(
                [0], [0], marker="o", color="w", markerfacecolor="none",
                markeredgecolor=_TRIGGER_COLOR, markeredgewidth=1.3, markersize=7,
                linewidth=0, label="ring: paired to a hidden trigger strand"))
        _ncol = min(len(handles), legend_ncol)
        if legend_below_figure:
            fig.legend(handles=handles, loc="lower center", ncol=_ncol, fontsize=legend_fontsize,
                       frameon=False, handletextpad=0.3, columnspacing=0.8)
            _rows = -(-len(handles) // _ncol)
            _strip = (_rows * legend_fontsize * 1.9 / 72.0) / figsize[1] + 0.02
            plt.tight_layout(rect=(0.0, _strip, 1.0, 0.96))
            return _png_from_fig(fig)
        ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.03),
                  ncol=_ncol, fontsize=legend_fontsize, frameon=False,
                  handletextpad=0.3, columnspacing=0.8)
    plt.tight_layout()
    return _png_from_fig(fig)


def bpp_heatmap_png(sequence, matrix, title, boundaries=(), figsize=(3.4, 3.1)):
    """Base-pair-probability heatmap -- labelled colourbar, numbered (position) axis
    ticks.

    ``boundaries``: ``(position, color)`` pairs -- one dotted CROSSHAIR (a horizontal AND
    a vertical line) per real strand junction present in this matrix's ``&``-joined
    frame, each in its own strand's color; a position in a 2-D matrix needs both axes
    drawn to be legible. A matrix with no second strand (an alone/OFF fold) passes no
    boundaries, since there is no junction to mark.
    """
    arr = np.array(matrix)
    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    im = ax.imshow(arr, cmap="magma", vmin=0.0, vmax=1.0, origin="upper")
    for position, color in boundaries:
        ax.axhline(position - 0.5, color=color, linewidth=1.1, linestyle=":")
        ax.axvline(position - 0.5, color=color, linewidth=1.1, linestyle=":")
    n = len(sequence)
    step = max(1, n // 12)
    ticks = list(range(0, n, step))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels([str(t) for t in ticks], fontsize=6.5, rotation=90)
    ax.set_yticklabels([str(t) for t in ticks], fontsize=6.5)
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("P(paired)", fontsize=7.5)
    cbar.ax.tick_params(labelsize=6.5)
    ax.set_title(title, fontsize=9, fontweight="bold")
    plt.tight_layout()
    return _png_from_fig(fig)


def layout_bar_png(segments, total_len, title):
    """Segmented domain-layout bar. `segments` is a list of
    (start, label, length, color, sequence) -- one bar for a single switch's one
    hairpin, or both hairpins of an AND construct in one fused-frame bar."""
    fig, ax = plt.subplots(figsize=(9.6, 2.0), dpi=130)
    for start, label, length, color, seq in sorted(segments, key=lambda s: s[0]):
        ax.barh(0, length, left=start, height=0.6, color=color, edgecolor="white", linewidth=1.2)
        if length / total_len > 0.02:
            ax.text(start + length / 2, 0, str(length), ha="center", va="center",
                    fontsize=7.5, fontweight="bold", color="white")
            ax.text(start + length / 2, -0.55, label, ha="center", va="top",
                    fontsize=6.6, color="#46524B")
            ax.text(start + length / 2, -1.05, seq, ha="center", va="top",
                    fontsize=5.0, family="monospace", color="#6B7870")
    ax.set_xlim(0, total_len)
    ax.set_ylim(-1.65, 0.5)
    ax.axis("off")
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    plt.tight_layout()
    return _png_from_fig(fig)


_REPORT_CSS = """
  :root {
    --scr-ground: #F6F8F4; --scr-surface: #FFFFFF; --scr-surface-2: #EEF2EA;
    --scr-ink: #131A16; --scr-ink-2: #46524B; --scr-ink-3: #75837B; --scr-line: #DCE4D8;
    --scr-accent-glow: #E4F7EA; --scr-accent-ink: #146134;
    --scr-structural: #4C6E8C;
    --scr-ok-bg: #D5F0DC; --scr-bad-bg: #F8D7D7;
    --scr-shadow: 0 1px 2px rgba(19,26,22,0.04), 0 8px 24px -12px rgba(19,26,22,0.12);
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      --scr-ground: #0E1512; --scr-surface: #141D18; --scr-surface-2: #1A2620;
      --scr-ink: #E9F2EC; --scr-ink-2: #AEC0B6; --scr-ink-3: #7C8D83; --scr-line: #26362D;
      --scr-accent-glow: #163524; --scr-accent-ink: #7DF0A2;
      --scr-structural: #8FB3D6;
      --scr-ok-bg: #1D4A2C; --scr-bad-bg: #5A2323;
      --scr-shadow: 0 1px 2px rgba(0,0,0,0.3), 0 8px 24px -12px rgba(0,0,0,0.5);
    }
  }
  * { box-sizing: border-box; }
  body.scr { margin: 0; background: var(--scr-ground); color: var(--scr-ink);
    font-family: "IBM Plex Sans", system-ui, sans-serif; padding: 0 20px; }
  .scr-mono { font-family: "IBM Plex Mono", ui-monospace, monospace; font-variant-numeric: tabular-nums; }
  .scr-wrap { max-width: 1040px; margin: 0 auto; padding-block: 44px 72px; }
  .scr-wrap h1 { font-size: clamp(26px, 4vw, 34px); font-weight: 700; line-height: 1.15;
    letter-spacing: -0.01em; margin: 0 0 8px; text-wrap: balance; }
  .scr-eyebrow { font-family: "IBM Plex Mono", monospace; font-size: 12px; letter-spacing: 0.08em;
    text-transform: uppercase; color: var(--scr-structural); margin: 0 0 10px; }
  .scr-dek { font-size: 15px; line-height: 1.6; color: var(--scr-ink-2); max-width: 78ch; margin: 0 0 32px; }
  .scr-dek ul.scr-intro { margin: 0; padding-left: 20px; }
  .scr-dek ul.scr-intro li { margin-bottom: 5px; }
  .scr-note { font-size: 12px; line-height: 1.5; color: var(--scr-ink-3); margin: 0 0 12px; max-width: 90ch; }
  .scr-ml { display: grid; grid-template-columns: 320px 1fr; gap: 2px 14px;
    font-size: 13px; line-height: 1.55; padding: 1px 0; }
  .scr-ml .k { color: var(--scr-ink-3); }
  .scr-ml .v { font-family: "IBM Plex Mono", monospace; color: var(--scr-ink); font-variant-numeric: tabular-nums; }
  .scr-metric-lines { margin: 0 0 12px; padding: 8px 12px; background: var(--scr-surface-2); border-radius: 8px; }
  .scr-table-wrap th small.scr-sub { display: block; font-size: 9px; font-weight: 400; text-transform: none;
    letter-spacing: 0; white-space: nowrap; margin-top: 2px; }
  .scr-table-wrap td.num small.scr-line { white-space: nowrap; font-size: 10.5px; color: var(--scr-ink-3); }
  .scr-table-wrap td.plan-ok { background: var(--scr-ok-bg); }
  .scr-table-wrap td.plan-bad { background: var(--scr-bad-bg); }
  .scr-structure-grid.scr-grid-2 { grid-template-columns: repeat(2, 1fr); }
  .scr-wrap section { margin-bottom: 40px; }
  .scr-wrap h2 { font-size: 20px; font-weight: 700; letter-spacing: -0.01em; color: var(--scr-ink);
    margin: 0 0 12px; padding-bottom: 8px; border-bottom: 2px solid var(--scr-line); }
  .scr-wrap h3 { font-family: "IBM Plex Mono", monospace; font-size: 14px; font-weight: 600;
    color: var(--scr-accent-ink); background: var(--scr-accent-glow); display: inline-block;
    padding: 4px 10px; border-radius: 6px; margin: 0 0 14px; }
  .scr-p { font-size: 13.5px; line-height: 1.7; color: var(--scr-ink-2); margin: 0 0 10px; max-width: 82ch; }
  .scr-table-wrap { overflow-x: auto; border: 1px solid var(--scr-line); border-radius: 10px;
    background: var(--scr-surface); box-shadow: var(--scr-shadow); margin-bottom: 8px; }
  .scr-table-wrap table { width: 100%; border-collapse: collapse; font-size: 12px; }
  .scr-table-summary table { min-width: 980px; }
  .scr-table-wrap th { text-align: left; font-size: 10px; letter-spacing: 0.05em; text-transform: uppercase;
    color: var(--scr-ink-3); font-weight: 600; padding: 8px 10px; border-bottom: 1px solid var(--scr-line);
    background: var(--scr-surface-2); white-space: nowrap; }
  .scr-table-wrap th.num { text-align: right; }
  .scr-table-wrap td { padding: 7px 10px; border-bottom: 1px solid var(--scr-line); color: var(--scr-ink-2); }
  .scr-table-wrap tr:last-child td { border-bottom: none; }
  /* Every numeric measurement column carries class="num" on BOTH its <th> and every
     <td>: a cell mixing a percentage with an nt-count (e.g. "100.0%" over "27.0/27 nt")
     is still ONE td, so the whole compound string right-aligns as a unit. tabular-nums
     keeps digit widths equal so numbers line up on their decimal point. */
  .scr-table-wrap td.num { text-align: right; font-family: "IBM Plex Mono", monospace;
    white-space: nowrap; font-variant-numeric: tabular-nums; }
  /* The two-line compound cells ("99.0%" over "17.8/18 nt") can be too long for a
     fixed-width decomposition column on one nowrap line; the secondary <small> line wraps
     inside its own cell instead of overflowing into the neighbour column
     (`table-layout:fixed` makes <colgroup> widths binding). */
  .scr-table-wrap td.num small { white-space: normal; display: block; }
  /* A long free-text cell (e.g. the single report's "+fake trigger" row when 0/N fakes
     qualified) must not use td.num: its white-space:nowrap runs the sentence off the
     page edge in a static PDF. This class gives left-aligned, wrapping text. */
  .scr-table-wrap td.scr-cell-note { text-align: left; font-family: inherit;
    white-space: normal; font-variant-numeric: normal; }
  .scr-candidate { background: var(--scr-surface); border: 1px solid var(--scr-line); border-radius: 12px;
    padding: 20px 22px; box-shadow: var(--scr-shadow); margin-bottom: 18px; }
  .scr-candidate-head { display: flex; align-items: baseline; justify-content: space-between;
    gap: 12px; margin-bottom: 12px; flex-wrap: wrap; }
  .scr-candidate-head h3 { margin: 0; }
  .scr-candidate-id { font-family: "IBM Plex Mono", monospace; font-size: 12px; color: var(--scr-ink-3); }
  .scr-img { max-width: 100%; height: auto; margin-bottom: 10px; }
  .scr-structure-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr));
    gap: 14px; margin-top: 10px; margin-bottom: 14px; }
  .scr-structure-grid figure { margin: 0; background: var(--scr-surface-2); border-radius: 8px;
    padding: 6px; text-align: center; }
  .scr-structure-grid img { width: 100%; height: auto; }
  .scr-structure-grid figcaption { font-size: 11px; color: var(--scr-ink-3); margin-top: 4px; }
  .scr-seq-block { font-size: 11px; line-height: 1.8; margin-bottom: 4px; word-break: break-all; }
  .scr-seq-block .label { display: inline-block; width: 120px; color: var(--scr-ink-3);
    font-family: "IBM Plex Mono", monospace; }
  footer.scr-footer { border-top: 1px solid var(--scr-line); padding-top: 18px; font-size: 11.5px;
    color: var(--scr-ink-3); }
  @media print {
    :root, :root:not([data-theme="light"]) {
      --scr-ground: #FFFFFF; --scr-surface: #FFFFFF; --scr-surface-2: #F3F5F1;
      --scr-ink: #131A16; --scr-ink-2: #3A443E; --scr-ink-3: #5C6660; --scr-line: #C9D2C4;
      --scr-ok-bg: #D5F0DC; --scr-bad-bg: #F8D7D7;
      --scr-shadow: none;
    }
    .scr-table-wrap td.plan-ok, .scr-table-wrap td.plan-bad {
      -webkit-print-color-adjust: exact; print-color-adjust: exact; }
    .scr-ml { grid-template-columns: 250px 1fr; font-size: 9.5px; }
    .scr-table-wrap th small.scr-sub { font-size: 6.5px; }
    .scr-table-wrap td.num small.scr-line { font-size: 7px; }
    body.scr { padding: 0 12px; }
    .scr-candidate, .scr-table-wrap { box-shadow: none; }
    .scr-candidate-head, .scr-seq-block, .scr-structure-grid figure, tr { break-inside: avoid; }
    thead { display: table-header-group; }
    /* The decomposition tables carry a percentage plus an nt-count per cell, so print gets
       a tighter font/padding to keep every column on the landscape A4 page
       (render_pdf.py renders @media print directly via Chromium). */
    .scr-table-wrap table { font-size: 8px; }
    .scr-table-wrap td, .scr-table-wrap th { padding: 3px 4px; }
    .scr-table-wrap th small { font-size: 6.8px; font-weight: 400; text-transform: none;
      letter-spacing: 0; display: block; }
    /* The energy table is only two columns with plenty of room -- exempt it from the
       squeeze above (rule ORDER wins here: same selector weight, placed after it). */
    .scr-table-energy table { font-size: 10.5px; }
    .scr-table-energy td, .scr-table-energy th { padding: 5px 8px; }
    /* The AND decomposition table (6 data columns + joint + state) needs a smaller base
       size even in print, on top of its <colgroup> widths. */
    .scr-table-and-decomp table { font-size: 7px; }
    .scr-table-and-decomp td, .scr-table-and-decomp th { padding: 2px 3px; }
  }
"""


def _num(value, places=2, suffix=""):
    """'--' for a measurement that is not there (CLAUDE.md sec 3), never a 0."""
    return "--" if value is None else f"{value:.{places}f}{suffix}"


def _pct(value):
    return "--" if value is None else f"{value:.1%}"


def _nt(fraction, length):
    # "{x}/{length} nt": the denominator is always shown.
    return "--" if fraction is None else f"{fraction * length:.1f}/{length} nt"


def _nt_pct(fraction, length):
    """Like ``_nt`` but with the mean % spelled out next to the nt-count -- e.g.
    "6.8/18 nt (37.8%)". Used for the FAKES displays."""
    return "--" if fraction is None else f"{fraction * length:.1f}/{length} nt ({fraction:.1%})"


def _pct_exact(mean, length):
    """Exact (partition-function) ensemble mean for the decomposition tables: the expected
    fraction of the span's nt, with the nt-count under it. ``--`` when it could not be
    computed (not measured is not zero -- CLAUDE.md sec 3)."""
    if mean is None:
        return "--"
    return f"{mean:.1%}<br><small>{mean * length:.1f}/{length} nt</small>"


def _mean_or_none(results, field):
    """Mean of ``field`` over the per-draw result dicts; ``None`` (not computed) if there
    are no draws or any draw lacks the value -- never a silent partial mean (CLAUDE.md
    sec 3). For the *_bpp fields this is the mean of the per-draw EXACT means."""
    vals = [r[field] for r in results]
    if not vals or any(v is None for v in vals):
        return None
    return sum(vals) / len(vals)


def _on_off_ratio(on, off, floor_pct):
    """``(ratio, on_minus_off)`` with ``ratio = on_pct / max(off_pct, floor_pct)``, both in
    PERCENTAGE POINTS (not 0-1 fractions). An OFF below ``floor_pct`` is replaced by it, so
    a measured OFF of 0.2% and one of 0.9% give the same ratio instead of one 4.5x larger;
    :func:`_off_floored` tells the report which rows that happened to. ``(None, None)`` if
    ON or OFF is ``None`` -- never treated as 0."""
    if on is None or off is None:
        return None, None
    return (on * 100.0) / max(off * 100.0, floor_pct), on - off


def _off_floored(off, floor_pct):
    """True when this row's OFF (a 0-1 fraction) is below the ratio's floor, i.e. the ratio
    used ``floor_pct`` instead of the measured OFF. ``None`` OFF -> False."""
    return off is not None and off * 100.0 < floor_pct


def _ratio_text(row):
    """Ratio as shown in the reports; a trailing ``*`` marks a row whose OFF was floored."""
    if row.get("ratio") is None:
        return "--"
    return f"{row['ratio']:.1f}" + ("*" if row.get("off_floored") else "")


def _ratio_floor_note(rows, floor_pct):
    """One-line note for under the summary table, or "" when no row was floored."""
    n = sum(1 for r in rows if r.get("off_floored"))
    if not n:
        return ""
    return (f'<p class="scr-p">* OFF is below {floor_pct:g}% in {n} of these {len(rows)} rows, so '
            f'{floor_pct:g}% was used as OFF in their ratio (ratio = ON% / max(OFF%, {floor_pct:g}%)).'
            f' The OFF column still shows the measured value.</p>')


def _render_pdf(html_path):
    """PDF next to ``html_path`` via ``toehold/render_pdf.py`` (found by walking up from
    the cwd); ``None`` with a message when the script is not found."""
    for base in (Path.cwd(), *Path.cwd().parents):
        script = base / "src/engine/gates/notebooks/toehold/render_pdf.py"
        if script.exists():
            break
    else:
        print("render_pdf.py not found -- HTML written, PDF skipped")
        return None
    pdf_path = html_path.with_suffix(".pdf")
    subprocess.run([sys.executable, str(script), str(html_path), str(pdf_path)], check=True)
    print(f"wrote {pdf_path}")
    return pdf_path


# ======================================================================================
# OFF (exact). A state's leak is its stem-loosening = 1 - mean_stem_closed_bpp: the share of
# the hairpin's stem_up that is NOT paired to its own stem_down, whatever the cause (a fake
# trigger, the other trigger, or the hairpin folding open on its own). OFF is the worst
# state. Per state, for every hairpin whose OWN real trigger is absent (``mask_side``
# names the side whose own trigger IS present and is therefore exempt).
# Without the mask, a working design's own intended trigger binding -- which necessarily
# displaces that side's stem_up from stem_down -- would score as a leak in every state
# holding any real trigger, collapsing OFF to ~100% for every candidate. A missing input
# (None) makes the state's leak None, and OFF None: never silently dropped to a lower
# OFF. (Trigger binding to the footprint is still SHOWN in the decomposition tables; it is
# no longer part of OFF.)
# ======================================================================================


def _single_badness(mean_stem_closed_bpp):
    """Leak of one single-switch state whose real trigger is absent (alone, or a fake / mean
    over fakes): its stem-loosening, ``1 - mean_stem_closed_bpp``. Returns
    ``(badness, criterion_label, terms)``; ``(None, None, {})`` if the input is missing.
    ``terms`` is the dict the max is taken over (``_off_breakdown_text`` renders it)."""
    if mean_stem_closed_bpp is None:
        return None, None, {}
    terms = {"stem-loosening": 1.0 - mean_stem_closed_bpp}
    criterion = max(terms, key=terms.get)
    return terms[criterion], criterion, terms


def _and_off_term(label, state, mask_side=None):
    """One ``(label, badness, mean_stem_closed_a_bpp, mean_stem_closed_b_bpp, criterion,
    terms)`` OFF candidate tuple for the AND scoring loop. The two means are the exact
    per-side stem_closed means of this state (display).

    ``mask_side`` is the side whose own REAL trigger is present in this state (``"a"``
    for +A and +A+fakeA, ``"b"`` for +B and +B+fakeB, ``None`` for alone/+fakeA/+fakeB):
    that side contributes no stem-loosening term, since its stem opening is the intended
    behaviour. Every unmasked side contributes its term, named ``stem-loosening (A)`` /
    ``stem-loosening (B)``. ``badness`` is
    ``None`` (with empty ``terms``) if any required value is missing."""
    terms = {}
    for side in ("a", "b"):
        if mask_side == side:
            continue
        sc = state[f"mean_stem_closed_{side}_bpp"]
        if sc is None:
            return (label, None, state["mean_stem_closed_a_bpp"], state["mean_stem_closed_b_bpp"],
                    None, {})
        terms[f"stem-loosening ({side.upper()})"] = 1.0 - sc
    criterion = max(terms, key=terms.get)
    return (label, terms[criterion], state["mean_stem_closed_a_bpp"],
            state["mean_stem_closed_b_bpp"], criterion, terms)


# ======================================================================================
# OFF display: every term ``_single_badness``/``_and_off_term`` considered, winner first,
# from the same ``terms`` dict they return -- pure formatting of numbers already computed.
# ======================================================================================


def _off_term_label(term):
    """``(display_label, "exact")`` for one OFF terms key. Every OFF term is an exact
    partition-function mean now, so the kind is always ``"exact"``."""
    return term.replace("(", "").replace(")", ""), "exact"


def _metric_lines(items):
    """Card head block: one ``(label_html, value_html)`` per line, label column aligned."""
    return ('<div class="scr-metric-lines">'
            + "".join(f'<div class="scr-ml"><span class="k">{k}</span><span class="v">{v}</span></div>'
                      for k, v in items)
            + "</div>")


def _plan_class(metric, hairpin, present, value):
    """Shading class for one measured cell of the AND per-state table: ``"plan-ok"`` when
    the value agrees with the design's plan for that state, ``"plan-bad"`` when it does
    not, ``""`` (no colour) when ``value`` is None.

    ``present`` = the real triggers in the state, a subset of ``("A", "B")`` (fake and
    alone states: ``()`` -- a fake counts as no real trigger). Plan:
      * ``bound`` (trigger_bound of ``hairpin``): HIGH iff its own trigger is present;
      * ``closed`` (stem_closed of ``hairpin``): LOW iff its own trigger is present, else HIGH;
      * ``misfolded``: LOW in every state;
      * ``joint`` (``hairpin`` None; ``P(duplexes)``): HIGH whenever a real trigger is
        present (it is only ever passed a value for the +A, +B and +A+B states: the
        probability that every designed duplex of the triggers present is formed).
    HIGH means value >= OPEN_THRESHOLD, LOW means value <= 1 - OPEN_THRESHOLD; a value in
    neither zone does not agree with a HIGH or a LOW plan (plan-bad).
    """
    if value is None:
        return ""
    if metric == "bound":
        want_high = hairpin in present
    elif metric == "closed":
        want_high = hairpin not in present
    elif metric == "misfolded":
        want_high = False
    elif metric == "joint":
        want_high = bool(present)
    else:
        raise ValueError(metric)
    ok = value >= OPEN_THRESHOLD if want_high else value <= 1.0 - OPEN_THRESHOLD
    return "plan-ok" if ok else "plan-bad"


def _plan_legend_html():
    return (f"Cell shading (colouring only, never scored): green = as designed, red = not as "
            f"designed (HIGH means at least "
            f"{OPEN_THRESHOLD:.0%}, LOW means at most {1.0 - OPEN_THRESHOLD:.0%}; a value in "
            f"between counts as not as designed; no colour = not measured; P(duplexes) in the +A, +B and +A+B rows is coloured like the rest).")


def _off_breakdown_text(off_source, off_criterion, terms):
    """``"(OFF set by: {off_source}'s {off_criterion} -- {criterion} {pct} vs. ...)"``
    -- every term in ``terms`` (all in the same leak-fraction units as badness itself,
    high = bad), winner first, then every other term that was considered and lost.

    Uses a real unicode em dash directly, NOT the ``&mdash;`` HTML entity: the single card
    head wraps this function's return value in its own ``html.escape()``, which would
    turn the entity into literal text. A real character passes through unchanged.
    """
    if not terms:
        return f"(OFF set by: {html.escape(str(off_source))}'s {html.escape(str(off_criterion))})"
    ordered = [off_criterion] + [k for k in terms if k != off_criterion]
    parts = [f"{html.escape(k)} {_pct(terms[k])}" for k in ordered]
    return (f"(OFF set by: {html.escape(str(off_source))}'s {html.escape(str(off_criterion))} "
            f"— " + " vs. ".join(parts) + ")")


# ==================================================================================
# CSV-rank vs report-rank correlation. `pool` = single_scored_pool: one slim dict per
# candidate in the FULL single-switch scored pool (csv_rank = the CSV's
# postgen_global_rank, 1 = best; on/off from this notebook's own scoring).
# REPORT RANKING = position when the pool is sorted by `_single_sort_key` (ON - OFF in # percentage points, rounded to 1 decimal, descending, then dG ascending; 1 = best) -- the # SAME key the ranking in cell 12 uses. ON - OFF can tie at 1 decimal, and dG breaks # those ties; only candidates tied in BOTH the rounded ON - OFF and dG
# (or both dG None) share the AVERAGE of the positions they span (1.5, 1.5, 3 ...).
# csv_rank is average-ranked the same way (a no-op while the CSV ranks are unique). Both
# vectors are oriented 1 = best, so a POSITIVE correlation means the CSV order and this
# report's order agree. No scipy in this environment: numpy / pure python only.
# ==================================================================================
def _single_sort_key(on_minus_off, d_ee):
    """THE single-switch ordering (cell 12 sorts by it; the correlation ranks by it):
    ON - OFF (a 0-1 fraction, taken in percentage points rounded to 1 decimal, the precision
    the report shows) DESCENDING, then dG (d_ee = ensemble ee(switch+trigger) - ee(switch
    alone), kcal/mol) ASCENDING -- more negative = better. A candidate whose dG is None
    sorts LAST within its tie, and one whose ON - OFF is None (ON or OFF not defined)
    sorts last overall -- never treated as 0.0. Smaller key = better."""
    return (on_minus_off is None, 0.0 if on_minus_off is None else -round(on_minus_off * 100.0, 1),
            d_ee is None, 0.0 if d_ee is None else d_ee)


def _average_ranks(values):
    """1-based ranks of `values` (ascending: smallest = rank 1; any comparable items,
    e.g. tuples). Exact ties get the mean of the positions they occupy."""
    n = len(values)
    order = sorted(range(n), key=lambda i: values[i])
    ranks = [0.0] * n
    pos = 0
    while pos < n:
        end = pos
        while end + 1 < n and values[order[end + 1]] == values[order[pos]]:
            end += 1
        avg = (pos + end) / 2.0 + 1.0
        for k in range(pos, end + 1):
            ranks[order[k]] = avg
        pos = end + 1
    return ranks


def _spearman_rho(rank_x, rank_y):
    """Pearson correlation of the two (average-)rank vectors; None if either is constant."""
    x, y = np.asarray(rank_x, dtype=float), np.asarray(rank_y, dtype=float)
    if len(x) < 2 or x.std() == 0.0 or y.std() == 0.0:
        return None
    return float(np.corrcoef(x, y)[0, 1])


def _kendall_tau_b(x, y):
    """Kendall tau-b = (C - D) / sqrt((n0 - n1)(n0 - n2)), n0 = n(n-1)/2, n1/n2 = pairs
    tied in x / in y. O(n^2) -- fine for a ~300-candidate pool. None if undefined."""
    n = len(x)
    conc = disc = tied_x = tied_y = 0
    for i in range(n - 1):
        for j in range(i + 1, n):
            dx, dy = x[i] - x[j], y[i] - y[j]
            if dx == 0 and dy == 0:
                tied_x += 1
                tied_y += 1
            elif dx == 0:
                tied_x += 1
            elif dy == 0:
                tied_y += 1
            elif (dx > 0) == (dy > 0):
                conc += 1
            else:
                disc += 1
    n0 = n * (n - 1) / 2.0
    denom = ((n0 - tied_x) * (n0 - tied_y)) ** 0.5
    return None if denom == 0 else (conc - disc) / denom


def _csv_vs_report_rank_stats(pool):
    """rho / tau-b between csv_rank and the rank-by-(ON - OFF) over the whole pool, plus the
    two rank vectors and the number of candidates sharing their ON - OFF with another."""
    csv_r = _average_ranks([int(p["csv_rank"]) for p in pool])
    keys = [_single_sort_key(None if p["on"] is None or p["off"] is None else p["on"] - p["off"],
                             p.get("d_ee")) for p in pool]
    rep_r = _average_ranks(keys)
    counts, counts_both = {}, {}
    for k in keys:
        counts[k[:2]] = counts.get(k[:2], 0) + 1
        counts_both[k] = counts_both.get(k, 0) + 1
    return {
        "n": len(pool), "csv_rank": csv_r, "report_rank": rep_r,
        "rho": _spearman_rho(csv_r, rep_r), "tau": _kendall_tau_b(csv_r, rep_r),
        "n_tied": sum(c for c in counts.values() if c > 1),            # share a displayed ON - OFF
        "n_tied_both": sum(c for c in counts_both.values() if c > 1),  # ...and the same dG
    }


def csv_vs_dg_rank_png(pool, report_csv_ranks):
    """Scatter: x = CSV rank, y = rank by dG alone (``d_ee``, more negative = better, 1 = best
    at the TOP; candidates without a dG last, ties share an average rank). Same layout and
    statistics as :func:`csv_vs_report_rank_png`, with ON and OFF left out entirely.
    Returns (base64 PNG, stats dict)."""
    csv_r = _average_ranks([int(p["csv_rank"]) for p in pool])
    dg_r = _average_ranks([(p.get("d_ee") is None, p.get("d_ee") or 0.0) for p in pool])
    st = {"n": len(pool), "rho": _spearman_rho(csv_r, dg_r), "tau": _kendall_tau_b(csv_r, dg_r),
          "n_no_dg": sum(1 for p in pool if p.get("d_ee") is None)}
    hi = np.asarray([int(p["csv_rank"]) in report_csv_ranks for p in pool])
    xs, ys = np.asarray(csv_r), np.asarray(dg_r)
    fig, ax = plt.subplots(figsize=(6.4, 5.6))
    lim = max(st["n"], 1) + 0.5
    ax.plot([0.5, lim], [0.5, lim], color="#999999", lw=0.8, ls="--", alpha=0.5, zorder=1,
            label="y = x (identical order)")
    ax.scatter(xs[~hi], ys[~hi], s=16, color=_COLOR_A, alpha=0.55, linewidths=0, zorder=2,
               label=f"pool candidate (N = {st['n']})")
    ax.scatter(xs[hi], ys[hi], s=34, color=_TRIGGER_COLOR, edgecolors="white", linewidths=0.5,
               zorder=3, label=f"in this report (top {int(hi.sum())} by ON - OFF)")
    ax.set_xlim(0.5, lim)
    ax.set_ylim(lim, 0.5)   # inverted: rank 1 (best) at the TOP
    ax.set_xlabel("CSV rank (postgen_global_rank; 1 = best)")
    ax.set_ylabel("rank by dG alone (most negative = 1)")
    _f = lambda v: "n/a" if v is None else f"{v:.2f}"  # noqa: E731
    ax.set_title(f"CSV rank vs dG rank -- N = {st['n']}, Spearman rho = {_f(st['rho'])}, "
                 f"Kendall tau = {_f(st['tau'])}", fontsize=9)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, frameon=False)
    ax.grid(alpha=0.15)
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.2)
    return _png_from_fig(fig), st


def csv_vs_report_rank_png(pool, report_csv_ranks):
    """Scatter: x = CSV rank, y = rank by this report's ON - OFF (1 = best at TOP).
    `report_csv_ranks` = csv_rank values of the rows that made the report (highlighted).
    Returns (base64 PNG, stats dict)."""
    st = _csv_vs_report_rank_stats(pool)
    hi = [int(p["csv_rank"]) in report_csv_ranks for p in pool]
    xs, ys = np.asarray(st["csv_rank"]), np.asarray(st["report_rank"])
    hi = np.asarray(hi)
    fig, ax = plt.subplots(figsize=(6.4, 5.6))
    lim = max(st["n"], 1) + 0.5
    ax.plot([0.5, lim], [0.5, lim], color="#999999", lw=0.8, ls="--", alpha=0.5, zorder=1,
            label="y = x (identical order)")
    ax.scatter(xs[~hi], ys[~hi], s=16, color=_COLOR_A, alpha=0.55, linewidths=0, zorder=2,
               label=f"pool candidate (N = {st['n']})")
    ax.scatter(xs[hi], ys[hi], s=34, color=_TRIGGER_COLOR, edgecolors="white", linewidths=0.5,
               zorder=3, label=f"in this report (top {int(hi.sum())} by ON - OFF)")
    ax.set_xlim(0.5, lim)
    ax.set_ylim(lim, 0.5)   # inverted: rank 1 (best) at the TOP
    ax.set_xlabel("CSV rank (postgen_global_rank; 1 = best)")
    ax.set_ylabel("rank by this report's ON - OFF, ties by dG (1 = best)")
    def _f(v):
        return "n/a" if v is None else f"{v:.2f}"

    ax.set_title(f"CSV rank vs report rank -- N = {st['n']}, Spearman rho = {_f(st['rho'])}, "
                 f"Kendall tau = {_f(st['tau'])}\n({st['n_tied']} tied ON - OFF values, broken by dG)",
                 fontsize=9)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, frameon=False)
    ax.grid(alpha=0.15)
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.2)   # room for the legend under the axis label
    return _png_from_fig(fig), st

# ======================================================================================
# "How ON and OFF were computed" blocks of the candidate cards. Pure formatting of values
# already measured (nothing is folded here): ON = the designed pairs, the cost of forcing
# them and the probability they give; OFF = every OFF state's two terms and their max.
# ======================================================================================


def _designed_pair_counts_single(row):
    """``[("", n)]``: how many designed pairs ``designed_pairs`` forces for this single switch."""
    from _joint_state_parallel import designed_pairs

    d = row["design"]
    pairs = designed_pairs(d["switch"], ((d["domains"]["toehold"][0], d["trigger"], len(d["switch"])),))
    return [("", None if pairs is None else len(pairs))]


def _designed_pair_counts_and(row):
    """``[("A", n_a), ("B", n_b)]`` for an AND pair (same offsets as the folded strand string)."""
    from _joint_state_parallel import both_triggers_extra, designed_pairs

    built = row["built"]
    n_self = len(built["fused"])
    extra = both_triggers_extra(row["d5"]["trigger"], row["d3"]["trigger"])
    offset, position = {}, n_self
    for strand in extra:
        offset[strand] = position
        position += len(strand)
    counts = []
    for label, key, trigger in (("A", "footprint_5p", row["d5"]["trigger"]),
                                ("B", "footprint_3p", row["d3"]["trigger"])):
        pairs = designed_pairs(built["fused"], ((built[key][0], trigger, offset[trigger]),))
        counts.append((label, None if pairs is None else len(pairs)))
    return counts


def _on_numbers_html(p_on, pair_counts, bound_items, rt):
    """Rows of the ON table: forced pairs, P(duplexes), the cost of forcing them
    (-RT ln P = G_forced - G, kcal/mol) and the average footprint bound in the same state."""
    import math

    pairs_txt = " &middot; ".join(
        f"{label + ' ' if label else ''}{'--' if n is None else n}" for label, n in pair_counts)
    cost = None if p_on is None or p_on <= 0.0 else -rt * math.log(p_on)
    bound_txt = " &middot; ".join(
        f"{label + ' ' if label else ''}{_nt(fraction, length)}" for label, fraction, length in bound_items)
    return (
        '<div class="scr-table-wrap" style="max-width:640px;"><table><tbody>'
        f"<tr><td>designed pairs forced (every complementary position of the trigger)</td>"
        f'<td class="num">{pairs_txt}</td></tr>'
        f'<tr><td><b>P(duplexes)</b> = probability all those pairs are present at once</td>'
        f'<td class="num"><b>{_pct(p_on)}</b></td></tr>'
        f"<tr><td>cost of forcing them, G_forced &minus; G = &minus;RT&middot;ln P "
        f"(RT = {rt:.3f} kcal/mol)</td>"
        f'<td class="num">{"--" if cost is None else f"{cost:.2f} kcal/mol"}</td></tr>'
        f"<tr><td>average footprint nt bound, same state (whole ensemble, not only fully "
        f"formed structures)</td>"
        f'<td class="num">{bound_txt}</td></tr>'
        "</tbody></table></div>")


def _off_numbers_html(states, winner):
    """Table of every OFF state: its stem loosening (1 - stem_closed, per hairpin whose own trigger is absent) and the max of them (the state's leak). ``states`` = ``[(key, label, terms,
    badness)]`` (terms ``None`` = not measured); ``winner`` = the key whose badness is OFF.
    A term a state does not have is the exempt side (its own trigger is present)."""
    order = ["stem-loosening", "stem-loosening (A)", "stem-loosening (B)"]
    cols = [c for c in order if any(t and c in t for _, _, t, _ in states)]
    head = "".join(f'<th class="num">{html.escape(c)}</th>' for c in cols)
    body = ""
    for key, label, terms, badness in states:
        bold = key == winner
        cells = "".join(
            f'<td class="num">{_pct(terms.get(c)) if terms and c in terms else "exempt"}</td>'
            for c in cols)
        tag = ("<b>", "</b>") if bold else ("", "")
        body += (f"<tr><td>{tag[0]}{html.escape(label)}{tag[1]}</td>{cells}"
                 f'<td class="num">{tag[0]}{_pct(badness)}{tag[1]}</td></tr>')
    return (
        '<div class="scr-table-wrap" style="max-width:900px;"><table>'
        f'<thead><tr><th>OFF state (hairpin\'s own trigger absent)</th>{head}'
        '<th class="num">state leak = max</th></tr></thead>'
        f"<tbody>{body}</tbody></table></div>")


def structure_svg(points, sequence, structure, title, spans=(), domain_fill=(), visible_len=None,
                  construct_len=None, spacing=23.0, pad=34.0, radius=7.4, legend_ncol=3):
    """One folded structure as an inline SVG: every base a circle with its letter inside, the
    backbone through them and a rung on every base pair, filled by domain.

    Same semantics as :func:`structure_png` (same ``spans`` / ``domain_fill`` / ``visible_len`` /
    ``construct_len`` arguments, same ring rules), drawn as vector so the letters stay sharp at
    any zoom. ``points`` is ``FoldEngine.layout_coordinates(structure)``. Bases are drawn
    ``spacing`` apart (the median neighbour distance is scaled to it), so the canvas grows with
    the molecule instead of the letters shrinking. Layout idea (lettered circles, scaling by the
    median neighbour gap, flipped y) follows Offer's ``structure_figures.py``.
    """
    import statistics

    n_vis = len(sequence) if visible_len is None else visible_len
    n_construct = n_vis if construct_len is None else construct_len
    vis = points[:n_vis]
    gaps = [
        ((vis[i + 1][0] - vis[i][0]) ** 2 + (vis[i + 1][1] - vis[i][1]) ** 2) ** 0.5
        for i in range(n_vis - 1)
    ]
    unit = statistics.median([g for g in gaps if g > 0]) or 1.0
    k = spacing / unit
    raw = [(p[0] * k, p[1] * k) for p in vis]
    span_x = max(x for x, _ in raw) - min(x for x, _ in raw)
    span_y = max(y for _, y in raw) - min(y for _, y in raw)
    if span_y > 1.15 * span_x:
        # a tall layout is turned 90 degrees so it uses a landscape page's width and the
        # letters stay legible when the plot is fitted to the page
        raw = [(y, x) for x, y in raw]
    min_x = min(x for x, _ in raw)
    max_y = max(y for _, y in raw)
    top = 38.0  # title strip
    pts = [(x - min_x + pad, max_y - y + pad + top) for x, y in raw]
    width = max(x for x, _ in pts) + pad
    height = max(y for _, y in pts) + pad

    stack, partner = [], {}
    for i, ch in enumerate(structure):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            j = stack.pop()
            partner[i], partner[j] = j, i

    ink, muted = "#141413", "#5F5E5A"
    parts = []
    for i, j in partner.items():
        if i < j and j < n_vis:
            (x1, y1), (x2, y2) = pts[i], pts[j]
            parts.append(f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" '
                         f'stroke="#8899AA" stroke-width="1.3" opacity="0.8"/>')
    backbone = " ".join(f"{'M' if q == 0 else 'L'} {x:.1f} {y:.1f}" for q, (x, y) in enumerate(pts))
    parts.append(f'<path d="{backbone}" fill="none" stroke="#B9C2BC" stroke-width="1.6" '
                 f'stroke-linecap="round" stroke-linejoin="round"/>')

    drew_payload = drew_trigger = False
    for i, (x, y) in enumerate(pts):
        fill = next((c for s, e, c, _l in domain_fill if s <= i < e), _DEFAULT_FILL)
        ring = next((c for s, e, c in spans if s <= i < e), None)
        j = partner.get(i)
        if ring is None and j is not None and j >= n_vis:
            # dangling stub: paired to a base that is deliberately not drawn
            if j < n_construct:
                ring, drew_payload = _PAYLOAD_RING_COLOR, True
            else:
                ring, drew_trigger = _TRIGGER_COLOR, True
        stroke, sw = (ring, 2.6) if ring else ("#666666", 0.8)
        parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{radius}" fill="{fill}" '
                     f'stroke="{stroke}" stroke-width="{sw}"/>')
        parts.append(f'<text x="{x:.1f}" y="{y + 3.4:.1f}" text-anchor="middle" '
                     f'font-family="Helvetica,Arial,sans-serif" font-size="9.5" font-weight="600" '
                     f'fill="{ink}">{html.escape(sequence[i])}</text>')
    for idx, text, dx in ((0, "5&#8242;", -17), (n_vis - 1, "3&#8242;", 17)):
        x, y = pts[idx]
        parts.append(f'<text x="{x + dx:.1f}" y="{y + 4:.1f}" text-anchor="middle" '
                     f'font-family="Helvetica,Arial,sans-serif" font-size="12" font-weight="700" '
                     f'fill="{muted}">{text}</text>')

    # legend: unique (label, colour) domain entries, then one entry per ring colour drawn
    entries = []
    for _s, _e, c, label in domain_fill:
        if (label, c, "fill") not in entries:
            entries.append((label, c, "fill"))
    if drew_payload:
        entries.append(("ring: paired into this construct's own hidden tail", _PAYLOAD_RING_COLOR, "ring"))
    if drew_trigger:
        entries.append(("ring: paired to a hidden trigger strand", _TRIGGER_COLOR, "ring"))
    col_w = width / legend_ncol
    rows = -(-len(entries) // legend_ncol) if entries else 0
    legend_top = height + 6
    for n, (label, c, kind) in enumerate(entries):
        lx = (n % legend_ncol) * col_w + 14
        ly = legend_top + (n // legend_ncol) * 20 + 10
        if kind == "fill":
            parts.append(f'<circle cx="{lx + 5:.1f}" cy="{ly - 4:.1f}" r="5.5" fill="{c}" '
                         f'stroke="#666666" stroke-width="0.8"/>')
        else:
            parts.append(f'<circle cx="{lx + 5:.1f}" cy="{ly - 4:.1f}" r="5.5" fill="none" '
                         f'stroke="{c}" stroke-width="2.4"/>')
        parts.append(f'<text x="{lx + 16:.1f}" y="{ly:.1f}" font-family="Helvetica,Arial,sans-serif" '
                     f'font-size="11" fill="{ink}">{html.escape(label)}</text>')
    total_h = height + (rows * 20 + 14 if rows else 0)
    title_svg = (f'<text x="{width / 2:.1f}" y="24" text-anchor="middle" '
                 f'font-family="Helvetica,Arial,sans-serif" font-size="15" font-weight="700" '
                 f'fill="{ink}">{html.escape(title)}</text>')
    return (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width:.0f} {total_h:.0f}" '
            f'style="width:100%;height:auto;max-height:640px;display:block;" role="img">{title_svg}{"".join(parts)}</svg>')


# Part 1 — single switch (no AND)

One gene, one trigger, one hairpin. Every state below is folded as a real multi-strand
complex through `FoldEngine` (never MFE-only, never a tube-style proxy), and every number is
exact: `trigger_bound`/`stem_closed`/`misfolded` are partition-function means of the state's
strand string, and the ON probability `P(duplexes)` is the exact probability that all designed
footprint-trigger base pairs form at once (`FoldEngine.constrained_probability`;
and_eu_report_spec.md Step 2). Output:
`var/reports/single_switch_report_<timestamp>.html/.pdf`.


In [ ]:
# ==================================================================================
# SNAPSHOT LOAD (pure load; nothing is recomputed). Runs before Part 1 so that when
# SNAPSHOT_TO_LOAD is set, every "if SNAPSHOT_TO_LOAD is None:" heavy cell below (both
# Parts) skips itself and both report cells find their rows already populated.
#
# `_report_root` (where every report/snapshot file gets written, var/reports at the repo
# root) is resolved here too, once: the report cells and the snapshot export all need it.
# ==================================================================================
_report_root = Path.cwd()
for _base in (_report_root, *_report_root.parents):
    if (_base / "pyproject.toml").exists():
        _report_root = _base / "var" / "reports"
        break
_report_root.mkdir(parents=True, exist_ok=True)

if SNAPSHOT_TO_LOAD is None:
    print("SNAPSHOT_TO_LOAD is None -- no-op (normal full run; nothing restored).")
else:
    _snap_path = Path(SNAPSHOT_TO_LOAD)
    if not _snap_path.is_absolute():
        _snap_path = _report_root / _snap_path
    if not _snap_path.exists():
        raise FileNotFoundError(f"{_snap_path} does not exist")
    _snap = json.loads(_snap_path.read_text())
    single_report_rows = _snap["single_report_rows"]
    and_report_rows = _snap["and_report_rows"]
    # Whole scored pool (slim). A snapshot without it gives None, and the single report
    # then shows a note instead of the CSV-vs-report rank section.
    single_scored_pool = _snap.get("single_scored_pool")

    # JSON has no tuples: every (start, end) span came back as a 2-element list; put them
    # back to tuples so anything keyed or compared by span behaves as on a fresh run.
    def _spans_to_tuples(obj):
        if isinstance(obj, dict):
            return {k: (tuple(v) if isinstance(v, list) and len(v) == 2
                        and all(isinstance(x, int) for x in v) else _spans_to_tuples(v))
                    for k, v in obj.items()}
        if isinstance(obj, list):
            return [_spans_to_tuples(v) for v in obj]
        return obj

    single_report_rows = _spans_to_tuples(single_report_rows)
    and_report_rows = _spans_to_tuples(and_report_rows)
    print(f"restored snapshot {_snap_path.name} (exported {_snap['exported_at']}): "
          f"{len(single_report_rows)} single-switch, {len(and_report_rows)} AND"
          + (f", pool scores for {len(single_scored_pool)} candidates"
             if single_scored_pool else ", NO pool scores (older snapshot)"))


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # FILTER_HOMOPOLYMER_RUN: load the WHOLE file, filter, THEN cut to SINGLE_POOL_N --
    # filtering a pre-capped pool would shrink it instead of keeping the next-best
    # candidates that pass.
    _load_limit = None if FILTER_HOMOPOLYMER_RUN else SINGLE_POOL_N
    single_designs, single_problems = load_ranked_designs(
        CSV_PATH, folder, columns=COLUMNS, carry_metrics=CARRY_METRICS, limit=_load_limit,
    )
    for _line_no, _reason in single_problems[:10]:
        print(f"  skipped line {_line_no}: {_reason}")

    if FILTER_HOMOPOLYMER_RUN:
        _n_before = len(single_designs)
        single_designs = [
            d for d in single_designs
            if "toehold" not in d["domains"]
            or longest_homopolymer(
                d["switch"][d["domains"]["toehold"][0]:d["domains"]["toehold"][1]]
            )[1] <= MAX_HOMOPOLYMER_RUN
        ]
        print(f"  FILTER_HOMOPOLYMER_RUN: {_n_before} loaded -> {len(single_designs)} passed "
              f"-> top {min(len(single_designs), SINGLE_POOL_N)} of those kept")
        single_designs = single_designs[:SINGLE_POOL_N]

    print(f"loaded {len(single_designs)} designs from {CSV_PATH} "
          f"({len(single_problems)} rows skipped)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the single-switch design load.")


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # SINGLE-SWITCH PARALLEL SCREEN (exact only -- one partition function per state). For
    # every candidate: alone, +real trigger, and +each of up to SINGLE_N_FAKES distinct
    # fakes -- all states of all candidates submitted as ONE list to ONE
    # run_parallel_single_exact call (CLAUDE.md sec 5 / Trap 6: one process-pool call per
    # batch, never one spin-up per job).
    #
    # Fake selection: pick_fake_triggers accepts pool triggers sharing at most
    # FAKE_MAX_SEQ_OVERLAP_NT nt of identical (same-direction) sequence with the
    # candidate's real trigger. Not every candidate need find SINGLE_N_FAKES qualifying
    # fakes; whatever count IS found is used and reported, never padded.
    # ==============================================================================
    _t0 = time.time()
    _single_jobs = []
    _single_fakes_by_rank = {}

    for _i, _d in enumerate(single_designs):
        _fp = (_d["domains"]["toehold"][0], _d["domains"]["stem_up"][1])
        _su, _sd = _d["domains"]["stem_up"], _d["domains"]["stem_down"]
        _single_jobs.append(SingleExactJob(
            key=f"{_d['rank']}|alone", switch=_d["switch"], extra="",
            footprint=_fp, stem_up=_su, stem_down=_sd,
            temperature=TEMPERATURE_C,
        ))
        # P(duplexes) (p_on) is only defined for the switch + REAL trigger state.
        _single_jobs.append(SingleExactJob(
            key=f"{_d['rank']}|real", switch=_d["switch"], extra=_d["trigger"],
            footprint=_fp, stem_up=_su, stem_down=_sd,
            temperature=TEMPERATURE_C, with_joint=True,
        ))

        _fakes = pick_fake_triggers(
            (_d["trigger"],), {_d["rank"]}, single_designs, SINGLE_N_FAKES,
            start_offset=_i + 1, max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
        )  # fewer than SINGLE_N_FAKES if the pool runs out
        _single_fakes_by_rank[_d["rank"]] = _fakes
        for _fk, _wd in enumerate(_fakes):
            _single_jobs.append(SingleExactJob(
                key=f"{_d['rank']}|fake{_fk}", switch=_d["switch"], extra=_wd["trigger"],
                footprint=_fp, stem_up=_su, stem_down=_sd,
                temperature=TEMPERATURE_C,
            ))

    _n_short = sum(1 for _f in _single_fakes_by_rank.values() if len(_f) < SINGLE_N_FAKES)
    print(f"{len(single_designs)} candidates, {len(_single_jobs)} jobs "
          f"(2 base + up to {SINGLE_N_FAKES} fakes each); "
          f"{_n_short} candidates found fewer than {SINGLE_N_FAKES} qualifying fakes")

    single_decomp_by_key = run_parallel_single_exact(_single_jobs, max_workers=MAX_WORKERS)
    _elapsed = time.time() - _t0
    print(f"done in {_elapsed:.0f}s ({_elapsed / len(_single_jobs) * 1000:.1f} ms/job)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the single-switch parallel screen.")


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ON/OFF RATIO (spec Step 2 / Trap 5; all quantities exact):
    #   ON  = p_on of the +real-trigger state: the exact probability that every
    #         designed footprint-trigger base pair (every complementary position of the trigger
    #         against the footprint, `designed_pairs`) is present at once --
    #         FoldEngine.constrained_probability. None when the duplex keeps no pair.
    #   OFF = max(stem-loosening over: alone, mean over SINGLE_N_FAKES fakes) -- the worst
    #         across genuinely illegitimate STATES, never "1 - ON" (Trap 5). Stem-loosening =
    #         1 - mean_stem_closed_bpp (`_single_badness`, cell 7): the share of stem_up open,
    #         whatever the cause; an exact partition-function mean, and for the pooled fakes
    #         the mean of the per-draw exact means.
    #
    # ratio = on_pct / max(off_pct, OFF_FLOOR_PCT) in percentage points (`_on_off_ratio`, cell 7). If ON
    # (or OFF) is None the ratio is None and the row sorts last -- never treated as 0.
    # ==============================================================================
    single_scored = []
    for _d in single_designs:
        _rank = _d["rank"]
        _alone = single_decomp_by_key[f"{_rank}|alone"]
        _real = single_decomp_by_key[f"{_rank}|real"]
        _fakes = _single_fakes_by_rank[_rank]
        _fake_results = [single_decomp_by_key[f"{_rank}|fake{_fk}"] for _fk in range(len(_fakes))]

        _on = _real["p_on"]
        # Which ILLEGITIMATE state sets OFF (with that state's own exact
        # mean_stem_closed_bpp and the CRITERION that won inside it), alongside the max.
        _badness_alone, _crit_alone, _terms_alone = _single_badness(
            _alone["mean_stem_closed_bpp"])
        _off_candidates = [("alone", _badness_alone, _alone["mean_stem_closed_bpp"], _crit_alone, _terms_alone)]
        _fake_mean_stem_closed = None
        if _fake_results:
            _fake_mean_stem_closed = _mean_or_none(_fake_results, "mean_stem_closed_bpp")
            _badness_fakes, _crit_fakes, _terms_fakes = _single_badness(_fake_mean_stem_closed)
            _off_candidates.append(("fakes", _badness_fakes, _fake_mean_stem_closed, _crit_fakes, _terms_fakes))
        if any(t[1] is None for t in _off_candidates):
            _off_source, _off, _off_mean_stem_closed, _off_criterion, _off_terms = (
                None, None, None, None, {})
        else:
            _off_source, _off, _off_mean_stem_closed, _off_criterion, _off_terms = max(
                _off_candidates, key=lambda t: t[1])

        _ratio, _on_minus_off = _on_off_ratio(_on, _off, OFF_FLOOR_PCT)
        _off_floored_flag = _off_floored(_off, OFF_FLOOR_PCT)

        # dG for EVERY candidate (needed by the tie-break below): the same quantity the
        # report shows -- ensemble ee(switch+trigger) - ee(switch alone), kcal/mol, via
        # folder.partition (exact, cached; 3 lookups/candidate, main process).
        _ee_switch = folder.partition(_d["switch"])
        _ee_trigger = folder.partition(_d["trigger"])
        _ee_on = folder.partition(f"{_d['switch']}&{_d['trigger']}")
        _d_ee = (_ee_on - _ee_switch) if (_ee_on is not None and _ee_switch is not None) else None

        single_scored.append({
            "design": _d, "rank": _rank,
            "alone": _alone, "real": _real, "fakes": _fakes, "fake_results": _fake_results,
            "on": _on, "off": _off, "off_source": _off_source, "off_criterion": _off_criterion,
            "off_terms": _off_terms,
            "off_mean_stem_closed": _off_mean_stem_closed,
            "fake_mean_stem_closed": _fake_mean_stem_closed,
            "ratio": _ratio, "off_floored": _off_floored_flag, "on_minus_off": _on_minus_off,
            "ee_switch": _ee_switch, "ee_trigger": _ee_trigger,
            "ee_switch_trigger": _ee_on, "d_ee": _d_ee,
        })

    # ORDER: ON - OFF in percentage points, rounded to 1 decimal (what the report displays), # descending; candidates that show the same ON - OFF are ordered by dG ascending (more negative = better); dG None
    # sorts last within its tie. Same `_single_sort_key` (cell 7) the CSV-vs-report
    # correlation ranks by. This also decides which singles enter the AND grid.
    single_scored.sort(key=lambda r: _single_sort_key(r["on_minus_off"], r["d_ee"]))
    single_report_rows = single_scored[:SINGLE_REPORT_N]
    # Slim per-candidate scores for the WHOLE scored pool (CSV-rank vs report-rank
    # correlation in the single report); a snapshot restore sets the same variable (cell 9).
    single_scored_pool = [
        {"csv_rank": int(r["rank"]), "on": r["on"], "off": r["off"], "ratio": r["ratio"],
         "d_ee": r["d_ee"],
         "off_source": r["off_source"], "off_criterion": r["off_criterion"]}
        for r in single_scored
    ]
    print(f"{len(single_scored)} candidates scored; top {len(single_report_rows)} "
          f"(ranks {[r['rank'] for r in single_report_rows]}) go to the report")
    print(f"  best on-off {_pct(single_scored[0]['on_minus_off'])}, ratio {_num(single_scored[0]['ratio'], 1)}, "
          f"on={_pct(single_scored[0]['on'])}, off={_pct(single_scored[0]['off'])} "
          f"(off_source={single_scored[0]['off_source']}, off_criterion={single_scored[0]['off_criterion']})")
    print(f"  {sum(1 for r in single_scored if r['ratio'] is None)} candidates have no ratio "
          f"(ON undefined: no designed footprint-trigger pair) and sort last")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping single-switch scoring.")


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ENERGIES (ensemble free energy via FoldEngine.partition, kcal/mol, more negative =
    # more stable). Per-candidate energies (incl. dG) are computed for the WHOLE pool in
    # cell 12 (the dG tie-break needs them); this cell only adds the mean dG over the
    # fakes for the top SINGLE_REPORT_N candidates that reach the report. Plain cached
    # scalar lookups, so serial calls are fine (one FoldEngine, same cache).
    # ==============================================================================
    for _r in single_report_rows:
        _switch = _r["design"]["switch"]
        _ee_switch = _r["ee_switch"]
        _fake_d_ees = [
            folder.partition(f"{_switch}&{_wd['trigger']}") - _ee_switch
            for _wd in _r["fakes"]
        ]
        _r["mean_d_ee_fakes"] = sum(_fake_d_ees) / len(_fake_d_ees) if _fake_d_ees else None
    print(f"energies computed for {len(single_report_rows)} report candidates")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping single-switch energies.")


In [ ]:
# Images -- UNGATED (always runs): rebuilds from whichever rows exist, fresh or restored
# from a snapshot (a snapshot never stores images: base64 PNGs would bloat it for nothing a
# fresh render in seconds does not give back). _PAYLOAD_DOMAINS: cell 7.

for _r in single_report_rows:
    _d = _r["design"]
    _switch = _d["switch"]
    _domains = _d["domains"]
    _toehold_span = _domains["toehold"]

    # 2D structure + bpp matrix, OFF (alone) and ON (+real trigger) -- through Kozak
    # only (payload dropped), a real separate fold of the shorter sequence so no
    # dangling bonds into a cropped-out tail.
    _kept = {n: s for n, s in _domains.items() if n not in _PAYLOAD_DOMAINS}
    _display_end = max((s[1] for s in _kept.values()), default=len(_switch))
    _display_seq = _switch[:_display_end]
    # Every base in a known domain span gets that domain's own colour.
    _domain_fill = _domain_fill_from_domains(_kept, _DOMAIN_COLOR_A)

    _mfe_off = folder.mfe(_display_seq)
    _matrix_off = folder.base_pair_probabilities(_display_seq)
    _r["structure_off_png"] = structure_png(
        _display_seq, _mfe_off.structure, f"rank {_r['rank']} — OFF (alone, through Kozak)",
        spans=[(_toehold_span[0], _toehold_span[1], _COLOR_A)], domain_fill=_domain_fill,
    )
    # No second strand in the OFF fold -- no junction to mark, so no boundary crosshair.
    _r["matrix_off_png"] = bpp_heatmap_png(
        _display_seq, _matrix_off, "OFF: base-pair probabilities",
    )

    _bound_strands = f"{_display_seq}&{_d['trigger']}"
    _bound_concat = _display_seq + _d["trigger"]  # structure/bpp index this, "&" removed
    # The trigger strand gets its own colour (_TRIGGER_COLOR), ON-state image only.
    _domain_fill_on = [*_domain_fill, (len(_display_seq), len(_bound_concat), _TRIGGER_COLOR, "trigger")]
    _mfe_on = folder.mfe(_bound_strands)
    _matrix_on = folder.base_pair_probabilities(_bound_strands)
    _r["structure_on_png"] = structure_png(
        _bound_concat, _mfe_on.structure, f"rank {_r['rank']} — ON (+real trigger)",
        spans=[(_toehold_span[0], _toehold_span[1], _COLOR_A)], domain_fill=_domain_fill_on,
    )
    # One real strand junction (switch -> trigger) at x=y=len(switch), one crosshair.
    _r["matrix_on_png"] = bpp_heatmap_png(
        _bound_concat, _matrix_on, "ON: base-pair probabilities",
        boundaries=[(len(_display_seq), _COLOR_A)],
    )

    # Domain diagram: layout bar (image) + each domain's own sequence, from _kept (payload
    # excluded), consistent with the structure/matrix plots above.
    _segments = [
        (s, name, e - s, _DOMAIN_COLOR_A.get(name, "#B7C4BC"), _switch[s:e])
        for name, (s, e) in sorted(_kept.items(), key=lambda kv: kv[1][0])
        if e > s
    ]
    # Total switch length on the bar itself (the trigger is never part of `_switch`).
    _r["layout_png"] = layout_bar_png(
        _segments, len(_switch),
        f"rank {_r['rank']} — domain layout (switch length {len(_switch)} nt)",
    )
    _r["domain_rows"] = [(name, length, seq) for (_s, name, length, _c, seq) in _segments]

print(f"images built for {len(single_report_rows)} single-switch candidates")


In [ ]:
# ==================================================================================
# SINGLE-SWITCH REPORT -- HTML (+ PDF). One candidate card per report row: energies, the
# three-way state decomposition per state (all exact), P(duplexes) for the +real trigger
# state, nt of trigger hybridised / nt of stem still hybridised, probability matrices,
# domain diagram with per-domain sequence. Every metric gets a one-line explanation
# (spec Step 4).
# ==================================================================================

def render_single_report_html(rows):
    summary_trs = []
    for i, r in enumerate(rows, start=1):
        _d = r["design"]
        _fp_len = _d["domains"]["stem_up"][1] - _d["domains"]["toehold"][0]
        _su_len = _d["domains"]["stem_up"][1] - _d["domains"]["stem_up"][0]
        # ON / OFF summary cells: line 1 = the headline (ON: exact P(duplexes); OFF: the
        # worst state's badness), line 2 = the exact nt-counts behind it.
        _on_cell = (f'{_pct(r["on"])}<small class="scr-line">'
                    f'{_nt(r["real"]["mean_trigger_bound_bpp"], _fp_len)} bound</small>')
        _off_cell = (f'{_pct(r["off"])}<small class="scr-line">'
                     f'{html.escape(str(r["off_source"]))} &middot; {html.escape(str(r["off_criterion"]))}'
                     f'</small><small class="scr-line">stem {_nt(r["off_mean_stem_closed"], _su_len)}</small>')
        summary_trs.append(
            "<tr>"
            f"<td>{i}</td><td>{html.escape(str(r['rank']))}</td>"
            f'<td class="num">{_on_cell}</td>'
            f'<td class="num">{_off_cell}</td>'
            f'<td class="num">{_ratio_text(r)}</td>'
            f'<td class="num">{_pct(r["on_minus_off"])}</td>'
            f'<td class="num">{_num(r["d_ee"])}</td>'
            f'<td><a href="#single-{i}">detail &darr;</a></td>'
            "</tr>"
        )

    cards = []
    for i, r in enumerate(rows, start=1):
        _d = r["design"]
        _fp_len = _d["domains"]["stem_up"][1] - _d["domains"]["toehold"][0]
        _su_len = _d["domains"]["stem_up"][1] - _d["domains"]["stem_up"][0]
        _alone, _real = r["alone"], r["real"]
        _fake_results = r["fake_results"]
        # Mean of the per-draw exact means over the fake draws (None if any draw lacks it).
        _mean_fake_tb = _mean_or_none(_fake_results, "mean_trigger_bound_bpp")
        _mean_fake_sc = _mean_or_none(_fake_results, "mean_stem_closed_bpp")
        _mean_fake_mf = _mean_or_none(_fake_results, "mean_misfolded_bpp")

        decomp_rows = f"""
          <tr><td><b>alone</b> (no trigger)</td>
              <td class="num">{_pct(_alone["mean_trigger_bound_bpp"])}</td>
              <td class="num">{_pct(_alone["mean_stem_closed_bpp"])}</td>
              <td class="num">{_pct(_alone["mean_misfolded_bpp"])}</td>
              <td class="num">{_nt(_alone["mean_stem_closed_bpp"], _su_len)}</td>
              <td class="num">--</td></tr>
          <tr><td><b>+real trigger</b> (ON)</td>
              <td class="num">{_pct(_real["mean_trigger_bound_bpp"])}</td>
              <td class="num">{_pct(_real["mean_stem_closed_bpp"])}</td>
              <td class="num">{_pct(_real["mean_misfolded_bpp"])}</td>
              <td class="num">{_nt(_real["mean_trigger_bound_bpp"], _fp_len)}</td>
              <td class="num">{_pct(_real["p_on"])}</td></tr>
        """
        if _fake_results:
            decomp_rows += f"""
              <tr><td>+fake trigger (mean of {len(_fake_results)}/{SINGLE_N_FAKES})</td>
                  <td class="num">{_pct(_mean_fake_tb)}</td>
                  <td class="num">{_pct(_mean_fake_sc)}</td>
                  <td class="num">{_pct(_mean_fake_mf)}</td>
                  <td class="num">{_nt_pct(_mean_fake_tb, _fp_len)}</td>
                  <td class="num">--</td></tr>
            """
        else:
            # 0 qualifying fakes is a legitimate outcome (every pool trigger shares more
            # than FAKE_MAX_SEQ_OVERLAP_NT nt of identical sequence with the real
            # trigger) -- but it must say so, not silently render as a row of "--".
            decomp_rows += f"""
              <tr><td>+fake trigger</td>
                  <td class="scr-cell-note" colspan="5"><i>no qualifying fakes found (0/{SINGLE_N_FAKES}
                  available): every other pool trigger shares more than
                  {FAKE_MAX_SEQ_OVERLAP_NT} nt of identical sequence with this
                  candidate's real trigger, so no fake-based OFF check ran for it.</i></td></tr>
            """

        # "How ON and OFF were computed": every OFF state's terms and the ON pair/cost numbers.
        _single_states = []
        _b, _c, _t = _single_badness(_alone["mean_stem_closed_bpp"])
        _single_states.append(("alone", "alone (no trigger)", _t or None, _b))
        if _fake_results:
            _b, _c, _t = _single_badness(_mean_fake_sc)
            _single_states.append(("fakes", f"+fake trigger (mean of {len(_fake_results)})", _t or None, _b))
        _numbers_html = (
            "<p class=\"scr-p\"><b>How ON and OFF were computed.</b> ON = P(duplexes) = "
            "exp(&minus;(G_forced &minus; G)/RT), from two partition functions.</p>"
            + _on_numbers_html(r["on"], _designed_pair_counts_single(r),
                               [("", _real["mean_trigger_bound_bpp"], _fp_len)], folder.rt)
            + "<p class=\"scr-p\">OFF = the largest state leak below (bold = the state that sets "
              f"OFF = {_pct(r['off'])}).</p>"
            + _off_numbers_html(_single_states, r["off_source"]))

        energy_rows = f"""
          <tr><td>switch alone</td><td class="num">{_num(r["ee_switch"])}</td></tr>
          <tr><td>trigger alone</td><td class="num">{_num(r["ee_trigger"])}</td></tr>
          <tr><td>switch + trigger</td><td class="num">{_num(r["ee_switch_trigger"])}</td></tr>
          <tr><td><b>dG (switch+trigger - switch alone)</b></td><td class="num">{_num(r["d_ee"])}</td></tr>
          <tr><td>mean dG across {len(r["fakes"])} fakes</td><td class="num">{_num(r["mean_d_ee_fakes"])}</td></tr>
        """

        # The switch sequence first, in one piece (5' -> 3') and only through the Kozak (the
        # reporter-gene payload is folded but not shown), then the same sequence cut into its
        # domains below.
        _full_seq = r["design"]["switch"][:r["design"]["domains"]["kozak"][1]]
        domain_block = (
            f'<div class="scr-seq-block"><span class="label">Switch sequence through the Kozak ({len(_full_seq)} nt, 5\' to 3\')</span>'
            f'<span class="scr-mono" style="word-break:break-all;">{html.escape(_full_seq)}</span></div>'
        ) + "".join(
            f'<div class="scr-seq-block"><span class="label">{html.escape(name)} ({length} nt)</span>'
            f'<span class="scr-mono">{html.escape(seq)}</span></div>'
            for name, length, seq in r["domain_rows"]
        )

        # Which state and hairpin set OFF for THIS candidate,
        # with every term considered (`_off_breakdown_text`, cell 7), not just the winner.
        _off_note = _off_breakdown_text(r["off_source"], r["off_criterion"], r.get("off_terms"))

        cards.append(f"""
        <div class="scr-candidate" id="single-{i}">
          <div class="scr-candidate-head">
            <h3>#{i} — rank {html.escape(str(r["rank"]))}</h3>
            <span class="scr-candidate-id">ON {_pct(r["on"])} ({_nt(r["real"]["mean_trigger_bound_bpp"], _fp_len)} bound) &middot; OFF {_pct(r["off"])} (stem {_nt(r["off_mean_stem_closed"], _su_len)}) {html.escape(_off_note)} &middot; ratio {_ratio_text(r)}</span>
          </div>
          <p class="scr-p"><b>Trigger ({len(_d["trigger"])} nt):</b> <span class="scr-mono">{html.escape(_d["trigger"])}</span><br>
             <b>FAKES &mdash; stem still closed with a fake trigger present</b> (mean over {len(_fake_results)} fakes): {_nt_pct(r["fake_mean_stem_closed"], _su_len)}</p>
          {_numbers_html}
          <p class="scr-p">Energies are ensemble &Delta;G in kcal/mol; more negative = more stable.</p>
          <div class="scr-table-wrap scr-table-energy" style="max-width:420px;"><table>
            <thead><tr><th>energy</th><th class="num">kcal/mol</th></tr></thead>
            <tbody>{energy_rows}</tbody>
          </table></div>
          <div class="scr-table-wrap"><table style="table-layout:fixed; width:100%;">
            <colgroup>
              <col style="width:20%"><col style="width:15%"><col style="width:15%">
              <col style="width:15%"><col style="width:17%"><col style="width:18%">
            </colgroup>
            <thead><tr><th>state</th><th class="num">trigger_bound<br><small>exact &middot; ON signal</small></th>
              <th class="num">stem_closed<br><small>exact</small></th>
              <th class="num">misfolded<br><small>exact</small></th>
              <th class="num">nt hybridised<br><small>exact</small></th>
              <th class="num">P(duplexes)<br><small>exact &middot; ON</small></th></tr></thead>
            <tbody>{decomp_rows}</tbody>
          </table></div>
          <img class="scr-img" src="data:image/png;base64,{r["layout_png"]}">
          <div class="scr-seq-block">{domain_block}</div>
          <div class="scr-structure-grid">
            <figure><img src="data:image/png;base64,{r["structure_off_png"]}"><figcaption>OFF structure</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["structure_on_png"]}">
              <figcaption>ON structure (trigger drawn after the switch for layout only --
                FoldEngine's "&amp;"-joined dimer convention; only the switch's OWN
                internal layout here is meaningful, not where the trigger is drawn)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_off_png"]}"><figcaption>OFF probability matrix</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_on_png"]}"><figcaption>ON probability matrix (dashed crosshair = switch/trigger boundary)</figcaption></figure>
          </div>
        </div>
        """)

    # Dynamic worked examples for the glossary -- real numbers from THIS render's own
    # #1 candidate, never a stale hand-typed number. All values are exact.
    _ex = rows[0]
    _ex_alone, _ex_real = _ex["alone"], _ex["real"]
    _ex_su = _ex["design"]["domains"]["stem_up"][1] - _ex["design"]["domains"]["stem_up"][0]
    _ex_fp = _ex["design"]["domains"]["stem_up"][1] - _ex["design"]["domains"]["toehold"][0]
    _worked_example_on = (
        f"e.g. for this report's own #1 candidate (rank {html.escape(str(_ex['rank']))}), "
        f"+real trigger state: P(duplexes) = {_pct(_ex_real['p_on'])}, while the exact mean "
        f"fraction of the footprint paired to the trigger is {_pct(_ex_real['mean_trigger_bound_bpp'])} "
        f"({_nt(_ex_real['mean_trigger_bound_bpp'], _ex_fp)})."
    )
    _worked_example_stem = (
        f"e.g. rank {html.escape(str(_ex['rank']))}, alone state (stem_up = {_ex_su} nt): "
        f"{_pct(_ex_alone['mean_stem_closed_bpp'])} means that, over the whole Boltzmann "
        f"ensemble, {_nt(_ex_alone['mean_stem_closed_bpp'], _ex_su)} of stem_up is expected "
        f"to be paired to stem_down."
    )
    _worked_example_mf = (
        f"e.g. rank {html.escape(str(_ex['rank']))}, alone state (footprint = {_ex_fp} nt): "
        f"{_pct(_ex_alone['mean_misfolded_bpp'])} means "
        f"{_nt(_ex_alone['mean_misfolded_bpp'], _ex_fp)} of the footprint is expected, "
        f"over the ensemble, to be paired to something other than its own stem_down."
    )

    return f"""<!DOCTYPE html><html><head><meta charset="utf-8">
    <title>Single-switch ON/OFF report</title><style>{_REPORT_CSS}</style></head>
    <body class="scr"><div class="scr-wrap">
      <p class="scr-eyebrow">CERNAL · eukaryotic toehold switch</p>
      <h1>Single-switch ON/OFF report</h1>
      <p class="scr-dek">Source: <span class="scr-mono">{html.escape(CSV_PATH)}</span>.
        {SINGLE_POOL_N} candidates screened, top {len(rows)} shown, ranked by ON/OFF
        ratio. Folding at {TEMPERATURE_C:.1f} &deg;C. <b>Method:</b> every value in this
        report is an exact partition-function quantity; nothing is sampled.
        <code>trigger_bound</code>, <code>stem_closed</code> and <code>misfolded</code> are
        exact ensemble means (<code>FoldEngine.base_pair_probabilities</code>): each is the
        expected fraction of that span's nt, e.g. "99.0%" of an 18 nt stem_up = 17.8 nt
        expected paired. <b>ON</b> is <code>P(duplexes)</code>, the exact probability
        (<code>FoldEngine.constrained_probability</code>) that every designed
        footprint-trigger base pair is present at once. <b>OFF</b> is the worst badness
        across {{alone, mean of up to {SINGLE_N_FAKES} off-target fakes}}, where a state's leak is its stem-loosening: the share of stem_up that is open (see the Glossary). <b>Ratio</b> = ON% / max(OFF%, {OFF_FLOOR_PCT:g}%), in percentage-point units. The
        green/red cell shading, where present, uses OPEN_THRESHOLD ({OPEN_THRESHOLD:.0%})
        only as a HIGH/LOW cutoff and never enters a score.</p>
      <section>
        <h2>Glossary</h2>
        <div class="scr-table-wrap"><table>
          <tbody>
            <tr><td style="width:130px"><b>ON</b> / <b>P(duplexes)</b></td><td>Exact probability
              that ALL designed footprint-trigger base pairs of the +real-trigger state are
              present in the same structure of the Boltzmann ensemble
              (<code>exp(-(G_forced - G)/RT)</code> from two partition functions). The
              designed pairs are every complementary position of the trigger against its footprint (the
              trigger aligned antiparallel from the footprint start; a designed mismatch is skipped); the
              event is strict: every one of those pairs must be present, so a duplex that is
              mostly formed but frays at one pair counts only for the structures that keep all
              of them. It is <code>--</code> (not zero) when the duplex has no designed pair.
              It cannot exceed the probability of any single one of its designed pairs.
              {_worked_example_on}</td></tr>
            <tr><td><b>OFF</b></td><td>the WORST <i>badness</i> across every illegitimate
              state checked (alone, mean over fake triggers) -- never "1 - ON". A state's leak = its stem-loosening (<code>1 &minus; stem_closed</code>, the exact expected fraction of stem_up NOT paired to stem_down), whatever the cause: a fake trigger, the other trigger, or the hairpin opening on its own. Every candidate's own card lists the leak of each state and which one set its OFF.</td></tr>
            <tr><td><b>ratio</b></td><td>ON% / max(OFF%, {OFF_FLOOR_PCT:g}%), in percentage-point units; an OFF below {OFF_FLOOR_PCT:g}% is replaced by {OFF_FLOOR_PCT:g}% in the ratio (the row is marked *) --
              higher is better; OFF=0 still gives a clean, finite value. Candidates are
              ordered by ON &minus; OFF as displayed (1 decimal); equal displayed values are ordered
              by dG, more negative first.</td></tr>
            <tr><td><b>Spearman rho / Kendall tau-b</b></td><td>rank correlations between the
              CSV's own ordering (postgen_global_rank) and the ordering by this report's ON &minus; OFF (ties broken by dG, more negative first), over the whole scored pool, both with 1 = best. +1 means the two orders
              agree exactly, 0 no relationship, &minus;1 fully reversed; rho correlates the rank
              numbers, tau-b counts concordant minus discordant candidate pairs (tie-corrected).</td></tr>
            <tr><td><b>dG</b></td><td>ensemble free-energy difference (switch+trigger minus
              switch alone), kcal/mol -- more negative means trigger binding stabilises the
              complex more.</td></tr>
            <tr><td><b>FAKES</b> (in each card)</td><td>an off-target check: a trigger NOT meant for this
              switch (a pool trigger sharing at most {FAKE_MAX_SEQ_OVERLAP_NT} nt of identical
              sequence with this switch's real trigger),
              checked for whether it accidentally opens the switch anyway. Shown as the mean
              nt of stem_up still bound to stem_down across up to {SINGLE_N_FAKES} fake-trigger
              draws, WITH that mean expressed as a % of stem_up right next to it (e.g. "6.8/18
              nt (37.8%)") -- the ideal is the FULL stem_up length (100%), so a reader can
              directly compare the shown % against OPEN_THRESHOLD without doing the division
              themselves. "--" means no qualifying fake could be found for this candidate
              under the {FAKE_MAX_SEQ_OVERLAP_NT} nt identical-sequence rule (see the note in that candidate's own
              card), not a failed measurement.</td></tr>
            <tr><td><b>trigger_bound</b></td><td>Exact expected fraction of the footprint (toehold+stem_up)
              paired to an EXTERNAL strand, from the partition function. A fraction of the
              FOOTPRINT length, so it also counts footprint positions the trigger does not cover; it is an
              average, not directly comparable with P(duplexes).</td></tr>
            <tr><td><b>stem_closed</b></td><td>Exact ensemble probability (partition
              function): the expected fraction of stem_up paired specifically
              to stem_down -- the intended OFF hairpin actually shut. Computed as, for each
              stem_up position, the total probability that it pairs to a position inside
              stem_down, averaged over stem_up. A fraction of the STEM_UP length alone (not
              the whole footprint); the nt-count shown is that fraction times the length.
              {_worked_example_stem}</td></tr>
            <tr><td><b>misfolded</b></td><td>Exact ensemble probability (partition
              function): the expected fraction of the footprint paired to
              something that is neither an external strand nor its own stem_down -- a
              structural risk signal, not a measured expression leak. A fraction of the
              FOOTPRINT length, same base as trigger_bound. {_worked_example_mf}</td></tr>
          </tbody>
        </table></div>
      </section>
      <section>
        <h2>Summary</h2>
        <p class="scr-p"><b>How to read ON and OFF.</b>
          <b>ON&nbsp;%</b> is P(duplexes): the probability, over the whole Boltzmann ensemble, that
          every designed footprint-trigger pair is formed at the same time (the share of molecules
          fully switched on). The small line under it is the average number of footprint
          nucleotides bound, out of the footprint length.
          <b>OFF&nbsp;%</b> is the leak in the worst OFF state, a state where the hairpin's own
          trigger is absent (alone, or with a fake trigger). It is the share of stem_up that is open, i.e. NOT paired to its own stem_down (stem loosening = 1 &minus; stem closed), whatever the cause: a fake trigger, the other trigger, or the hairpin opening on its own. It is an
          average share of nucleotides, not the probability of one event, so ON and OFF are not the
          same kind of number. The small lines under it name the state and the term (stem loosening), and give the
          stem nucleotides still closed there. <b>ratio</b> = ON&nbsp;% / max(OFF&nbsp;%,
          {OFF_FLOOR_PCT:g}&nbsp;%).</p>
        <div class="scr-table-wrap scr-table-summary"><table>
          <thead><tr><th>#</th><th>rank</th><th class="num">ON<small class="scr-sub">P(all designed pairs formed)<br>avg footprint nt bound</small></th>
            <th class="num">OFF<small class="scr-sub">leak in worst OFF state<br>state &middot; term<br>stem nt still closed there</small></th><th class="num">ratio</th>
            <th class="num">ON&minus;OFF</th><th class="num">dG</th><th></th></tr></thead>
          <tbody>{"".join(summary_trs)}</tbody>
        </table></div>
        {_ratio_floor_note(rows, OFF_FLOOR_PCT)}
        <p class="scr-p">See the Glossary above for what every column means. The fake-trigger check
          (FAKES) is in each candidate's card.</p>
      </section>
      {_corr_section_html}
      <section><h2>Candidates</h2>{"".join(cards)}</section>
      <footer class="scr-footer">Generated {datetime.datetime.now().isoformat(timespec="seconds")}
        · OPEN_THRESHOLD={OPEN_THRESHOLD} (plan colouring only)
        · all values exact (partition function); nothing sampled
        · FoldEngine {folder.versions()}</footer>
    </div></body></html>"""


_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
# CSV-rank vs report-rank section: needs the WHOLE scored pool, which a fresh run (cell 12)
# and a snapshot restore (cell 9) both put in `single_scored_pool`.
if single_scored_pool:
    _corr_b64, _corr_st = csv_vs_report_rank_png(
        single_scored_pool, {int(r["rank"]) for r in single_report_rows})
    _corr_png_path = _report_root / f"csv_vs_report_rank_{_ts}.png"
    _corr_png_path.write_bytes(base64.b64decode(_corr_b64))
    print(f"wrote CSV-vs-report rank plot -> {_corr_png_path}")
    _dg_b64, _dg_st = csv_vs_dg_rank_png(
        single_scored_pool, {int(r["rank"]) for r in single_report_rows})
    _dg_png_path = _report_root / f"csv_vs_dg_rank_{_ts}.png"
    _dg_png_path.write_bytes(base64.b64decode(_dg_b64))
    print(f"wrote CSV-vs-dG rank plot -> {_dg_png_path}")
    def _cf(v):
        return "n/a" if v is None else f"{v:.2f}"

    _corr_section_html = f"""<section>
        <h2>CSV ranking vs this report's ranking</h2>
        <p class="scr-p">Each dot is one of the {_corr_st["n"]} candidates scored (SINGLE_POOL_N =
          {SINGLE_POOL_N}); x is its position in the CSV (postgen_global_rank, 1 = best), y its
          position when the same pool is sorted by this report's ON &minus; OFF (1 = best, at the
          top; candidates showing the same ratio are ordered by dG, more negative first); dots on the dashed diagonal are ranked identically by both, and the red dots are
          the top {len(single_report_rows)} (SINGLE_REPORT_N = {SINGLE_REPORT_N}) shown in this report.</p>
        <img class="scr-img" style="max-width:100%;height:auto;" src="data:image/png;base64,{_corr_b64}">
        <p class="scr-p"><b>Spearman rho = {_cf(_corr_st["rho"])}</b> &middot;
          <b>Kendall tau-b = {_cf(_corr_st["tau"])}</b> over N = {_corr_st["n"]} candidates
          (+1 = same order, 0 = unrelated, &minus;1 = reversed). {_corr_st["n_tied"]} candidates share
          their displayed ON &minus; OFF with another (ordered by dG); only the {_corr_st["n_tied_both"]} tied in
          both ratio and dG share an average rank.</p>
        <h3>CSV ranking vs ranking by dG alone</h3>
        <p class="scr-p">Same pool and x axis, but y is the position when the pool is sorted by
          dG only (ee(switch+trigger) &minus; ee(switch) &minus; ee(trigger); more negative =
          rank 1, at the top), ignoring ON and OFF. It shows whether the CSV order follows the
          binding energy.</p>
        <img class="scr-img" style="max-width:100%;height:auto;" src="data:image/png;base64,{_dg_b64}">
        <p class="scr-p"><b>Spearman rho = {_cf(_dg_st["rho"])}</b> &middot;
          <b>Kendall tau-b = {_cf(_dg_st["tau"])}</b> over N = {_dg_st["n"]} candidates
          ({_dg_st["n_no_dg"]} without a dG, ranked last).</p>
      </section>"""
else:
    _corr_section_html = """<section>
        <h2>CSV ranking vs this report's ranking</h2>
        <p class="scr-p"><i>pool scores not in this snapshot; re-run the notebook to produce them.</i></p>
      </section>"""
    print("single_scored_pool missing -- CSV-vs-report rank plot skipped")

_single_html = render_single_report_html(single_report_rows)
single_report_html_path = _report_root / f"single_switch_report_{_ts}.html"
single_report_html_path.write_text(_single_html)
print(f"wrote {single_report_html_path.stat().st_size:,} bytes -> {single_report_html_path}")

single_report_pdf_path = _render_pdf(single_report_html_path)


# Part 2 — AND (two inputs)

`AND_GRID_N` top singles (by single-switch ON - OFF) fused into every ordered pair
(self-pairs excluded), both spacer variants. Each pair's 8 states (alone, +A, +B, +A+B,
+fakeA mean of up to `AND_N_FAKES`, +fakeB mean, +A+fakeA mean, +B+fakeB mean) are
decomposed the same three-way way as Part 1, from one exact pair-probability matrix per
state. ON is `P(duplexes)` of the +A+B state: the exact probability that BOTH designed
footprint-trigger duplexes are present in the same structure
(`FoldEngine.constrained_probability`) -- a joint event that is not the product of the two
marginal bound columns. Nothing is sampled. Output: `var/reports/and_report_<timestamp>.html/.pdf`.


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # AND GRID -- AND_GRID_N x itself, self-pairs excluded, both spacer variants.
    # Every job of every pair of every spacer variant is built first, then submitted
    # as ONE list to ONE run_parallel_and_decomp call (CLAUDE.md sec 5 / Trap 6).
    # ==============================================================================
    and_pool_designs = [r["design"] for r in single_scored[:AND_GRID_N]]
    print(f"AND pool: top {len(and_pool_designs)} singles by single-switch ON - OFF "
          f"(ranks {[d['rank'] for d in and_pool_designs]})")

    SPACER_CANDIDATES = generate_spacer_candidates(
        SPACER_CANDIDATE_POOL_SIZE, SPACER_MIN_LENGTH, SPACER_MAX_LENGTH, seed=SPACER_SEED,
    )

    # FAKES COME FROM THE FULL SINGLE POOL (single_designs, SINGLE_POOL_N candidates), not
    # only the AND_GRID_N pool the pairs are made from. A fake shares at most
    # FAKE_MAX_SEQ_OVERLAP_NT nt of identical sequence with EITHER real trigger of the pair.
    and_jobs = []
    and_built_by_key = {}  # key -> (d5, d3, built, spacer_label, spacer_seq, spacer_worst,
    #                              fakes_a, fakes_b)
    _and_t0 = time.time()
    _n_excluded_self = 0
    _n_excluded_trigger_overlap = 0
    _n_fuse_failed = 0

    for _spacer_label in SPACER_VARIANTS:
        for _d5 in and_pool_designs:
            for _d3 in and_pool_designs:
                if _d5["rank"] == _d3["rank"]:
                    _n_excluded_self += 1
                    continue
                # Trigger-vs-trigger overlap screen (complementarity with WOBBLE_MAX_FRACTION
                # AND same-direction identity, both capped at FAKE_MAX_SEQ_OVERLAP_NT; see
                # trigger_pair_too_similar): skips this (d5, d3) combo before any job is
                # built. Cheap sequence comparison, so repeated per spacer variant.
                if trigger_pair_too_similar(_d5["trigger"], _d3["trigger"],
                                            max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                                            wobble_max_fraction=WOBBLE_MAX_FRACTION):
                    _n_excluded_trigger_overlap += 1
                    continue
                if _spacer_label == "spacer":
                    _spacer_seq, _spacer_worst = pick_optimized_spacer(_d5, _d3, SPACER_CANDIDATES)
                else:
                    _spacer_seq, _spacer_worst = "", None
                _built, _problem = fuse(_d5, _d3, cut_before=CUT_5P_BEFORE, keep_after=KEEP_3P_AFTER,
                                        spacer=_spacer_seq)
                if _problem is not None:
                    _n_fuse_failed += 1
                    continue

                _pair_key = f"{_d5['rank']}+{_d3['rank']}|{_spacer_label}"
                _real_triggers = (_d5["trigger"], _d3["trigger"])
                _pair_ranks = {_d5["rank"], _d3["rank"]}
                _fakes_a = pick_fake_triggers(
                    _real_triggers, _pair_ranks, single_designs, AND_N_FAKES,
                    start_offset=0, max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                )
                _fakes_b = pick_fake_triggers(
                    _real_triggers, _pair_ranks, single_designs, AND_N_FAKES,
                    start_offset=1000, max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                )
                and_built_by_key[_pair_key] = (_d5, _d3, _built, _spacer_label, _spacer_seq,
                                                _spacer_worst, _fakes_a, _fakes_b)
                and_jobs.extend(and_pair_jobs(
                    _built, _pair_key, _d5["trigger"], _d3["trigger"], _fakes_a, _fakes_b,
                    temperature=TEMPERATURE_C,
                ))

    print(f"{len(and_built_by_key)} (pair, spacer) combos, {len(and_jobs)} jobs total "
          f"({_n_excluded_self} self-pairs excluded, {_n_excluded_trigger_overlap} excluded by "
          f"the trigger-vs-trigger overlap screen, {_n_fuse_failed} failed fuse()) -- "
          f"ONE pool.map call")
    and_decomp_by_key = run_parallel_and_decomp(and_jobs, max_workers=MAX_WORKERS)
    _and_elapsed = time.time() - _and_t0
    print(f"AND grid done in {_and_elapsed:.0f}s ({_and_elapsed / 60:.1f} min), "
          f"{_and_elapsed / len(and_jobs) * 1000:.1f} ms/job")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the AND grid.")


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ON/OFF RATIO for AND (spec Step 2 / Trap 5; all quantities exact):
    #   ON  = p_on of the "+A+B" state: the exact probability that BOTH designed
    #         footprint-trigger duplexes (the footprint pairs to external strands in that
    #         state's designed alignment) are present in the same structure --
    #         FoldEngine.constrained_probability. None if either duplex keeps no pair.
    #   OFF = max over the leak of {alone, +A, +B, mean(+fakeA), mean(+fakeB),
    #         mean(+A+fakeA), mean(+B+fakeB)} -- never "1 - ON" (Trap 5). A state's leak
    #         (`_and_off_term`/cell 7) is the max, over every hairpin whose OWN real trigger
    #         is absent in that state, of its stem-loosening (1 - stem_closed): the share of
    #         stem_up that is open, whatever the cause; exact partition-function means.
    #
    # ratio = on_pct / max(off_pct, OFF_FLOOR_PCT), percentage-POINT units (`_on_off_ratio`, cell 7),
    # same as the single-switch cell. If ON or OFF is None the ratio is None and the row
    # sorts last (never treated as 0).
    # ==============================================================================
    def _pooled_state(results):
        # Mean of the per-draw exact means, for the fields OFF scores and displays.
        return {
            f: _mean_or_none(results, f) for f in ("mean_stem_closed_a_bpp", "mean_stem_closed_b_bpp")
        }

    and_scored = []
    for _key, (_d5, _d3, _built, _spacer_label, _spacer_seq, _spacer_worst, _fakes_a, _fakes_b) in and_built_by_key.items():
        _alone = and_decomp_by_key[f"{_key}|alone"]
        _a = and_decomp_by_key[f"{_key}|+A"]
        _b = and_decomp_by_key[f"{_key}|+B"]
        _both = and_decomp_by_key[f"{_key}|+A+B"]
        _fakeA_results = [and_decomp_by_key[f"{_key}|+fakeA{_i}"] for _i in range(len(_fakes_a))]
        _fakeB_results = [and_decomp_by_key[f"{_key}|+fakeB{_i}"] for _i in range(len(_fakes_b))]
        _a_fakeA_results = [and_decomp_by_key[f"{_key}|+A+fakeA{_i}"] for _i in range(len(_fakes_a))]
        _b_fakeB_results = [and_decomp_by_key[f"{_key}|+B+fakeB{_i}"] for _i in range(len(_fakes_b))]

        # (label, badness, mean_stem_closed_a_bpp, mean_stem_closed_b_bpp, criterion, terms)
        # for every illegitimate state, built by `_and_off_term` (cell 7). mask_side
        # exempts the side whose own REAL trigger is present in the state (+A/+A+fakeA
        # mask "a"; +B/+B+fakeB mask "b"; alone/+fakeA/+fakeB mask neither); without it
        # OFF collapses to ~100% for every candidate (spec, OFF).
        _off_candidates = [
            _and_off_term("alone", _alone, mask_side=None),
            _and_off_term("+A", _a, mask_side="a"),
            _and_off_term("+B", _b, mask_side="b"),
        ]
        for _label, _results, _mask in (
            ("+fakeA", _fakeA_results, None), ("+fakeB", _fakeB_results, None),
            ("+A+fakeA", _a_fakeA_results, "a"), ("+B+fakeB", _b_fakeB_results, "b"),
        ):
            if not _results:
                continue
            _off_candidates.append(_and_off_term(_label, _pooled_state(_results), mask_side=_mask))
        if any(t[1] is None for t in _off_candidates):
            # a required exact value is missing: OFF (and the ratio) is not defined
            _off_source, _off, _off_mean_sc_a, _off_mean_sc_b, _off_criterion, _off_terms = (
                None, None, None, None, None, {})
        else:
            _off_source, _off, _off_mean_sc_a, _off_mean_sc_b, _off_criterion, _off_terms = max(
                _off_candidates, key=lambda t: t[1])

        # Per-side fake-check stem nt-count (fakeA against hairpin A's own stem, fakeB
        # against hairpin B's): mean of the exact mean_stem_closed_{a,b}_bpp over that
        # side's fake draws.
        _fakeA_mean_stem_closed = _mean_or_none(_fakeA_results, "mean_stem_closed_a_bpp")
        _fakeB_mean_stem_closed = _mean_or_none(_fakeB_results, "mean_stem_closed_b_bpp")

        _on = _both["p_on"]
        _ratio, _on_minus_off = _on_off_ratio(_on, _off, OFF_FLOOR_PCT)
        _off_floored_flag = _off_floored(_off, OFF_FLOOR_PCT)

        and_scored.append({
            "key": _key, "d5": _d5, "d3": _d3, "built": _built,
            "spacer_label": _spacer_label, "spacer_seq": _spacer_seq, "spacer_worst": _spacer_worst,
            "fakes_a": _fakes_a, "fakes_b": _fakes_b,
            "alone": _alone, "state_a": _a, "state_b": _b, "both": _both,
            "fakeA_results": _fakeA_results, "fakeB_results": _fakeB_results,
            "a_fakeA_results": _a_fakeA_results, "b_fakeB_results": _b_fakeB_results,
            "fakeA_mean_stem_closed": _fakeA_mean_stem_closed,
            "fakeB_mean_stem_closed": _fakeB_mean_stem_closed,
            "off_source": _off_source, "off_criterion": _off_criterion, "off_terms": _off_terms,
            "off_mean_stem_closed_a": _off_mean_sc_a,
            "off_mean_stem_closed_b": _off_mean_sc_b,
            "on": _on, "off": _off, "ratio": _ratio, "off_floored": _off_floored_flag,
            "on_minus_off": _on_minus_off,
        })

    # ranked by ON - OFF, descending (the ratio is still computed and shown); rows where ON
    # or OFF is undefined (on_minus_off None) last, never treated as 0
    and_scored.sort(key=lambda r: (
        r["on_minus_off"] is None, 0.0 if r["on_minus_off"] is None else -r["on_minus_off"]))
    and_report_rows = and_scored[:AND_REPORT_N]
    print(f"{len(and_scored)} (pair, spacer) combos scored ({sum(1 for r in and_scored if r['on_minus_off'] is None)} "
          f"without ON-OFF: ON or OFF undefined); top {len(and_report_rows)} go to the report")
    if and_scored:
        _top = and_scored[0]
        print(f"  best: {_top['d5']['rank']}+{_top['d3']['rank']} [{_top['spacer_label']}] "
              f"on-off={_pct(_top['on_minus_off'])}, ratio={_num(_top['ratio'], 1)}, on={_pct(_top['on'])}, off={_pct(_top['off'])} "
              f"(off_source={_top['off_source']}, off_criterion={_top['off_criterion']})")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping AND scoring.")


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ENERGIES for AND, top AND_REPORT_N only: alone, trigger A alone, trigger B
    # alone, +A, +B, +A+B, and Delta(ON) = ee_both - min(ee_off, ee_tA, ee_tB) (more
    # negative is better) -- measure()'s own d_ee_worst_alt IS this quantity, reused as-is
    # (CLAUDE.md sec 1).
    # ==============================================================================
    for _r in and_report_rows:
        _d5, _d3, _built = _r["d5"], _r["d3"], _r["built"]
        _dg = measure(_built, _d5["trigger"], _d3["trigger"], folder,
                      _d3["metrics"]["own_kozak_rc_score"], _d5["metrics"]["own_prefix_rc_score"])
        _r["dg"] = _dg
        _r["ee_trigger_a"] = folder.partition(_d5["trigger"])
        _r["ee_trigger_b"] = folder.partition(_d3["trigger"])
    print(f"energies computed for {len(and_report_rows)} AND report candidates")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping AND energies.")


In [ ]:
# Images -- UNGATED (always runs, same rationale as the single-switch images cell).
#
# Every structure plot is clipped (`visible_len`, the mechanism structure_png uses to hide
# a trigger strand) to the end of the LAST real kozak domain in the fused frame -- for this
# topology hairpin B's -- so the payload tail that fuse() keeps attached past kozak is not
# drawn as unexplained grey circles. Any real base pair whose partner falls outside that
# range still gets the dangling-ring treatment, so a bond into the hidden tail or into a
# trigger strand is never silently dropped.
# The two structure plots of the AND card are inline SVG (`structure_svg`, cell 7): every base
# a lettered circle, full card width. The matrix plots below stay PNG.
_AND_BPP_FIGSIZE = (5.2, 4.5)

for _r in and_report_rows:
    _built = _r["built"]
    _fused = _built["fused"]
    _fp5, _fp3 = _built["footprint_5p"], _built["footprint_3p"]
    _spans2 = [(_fp5[0], _fp5[1], _COLOR_A), (_fp3[0], _fp3[1], _COLOR_B)]

    # Per-domain fill colour for BOTH hairpins -- a doubled palette with a " (A)"/" (B)"
    # label suffix so the legend tells hairpin A's "toehold" apart from hairpin B's.
    # Payload domains are excluded via the shared _PAYLOAD_DOMAINS (cell 7).
    _kept5p = {n: s for n, s in _built["domains_5p"].items() if n not in _PAYLOAD_DOMAINS}
    _kept3p = {n: s for n, s in _built["domains_3p"].items() if n not in _PAYLOAD_DOMAINS}
    _domain_fill_and = (
        _domain_fill_from_domains(_kept5p, _DOMAIN_COLOR_A, " (A)")
        + _domain_fill_from_domains(_kept3p, _DOMAIN_COLOR_B, " (B)")
    )
    if _built.get("spacer_span"):
        _sp0, _sp1 = _built["spacer_span"]
        if _sp1 > _sp0:
            _domain_fill_and.append((_sp0, _sp1, _SPACER_COLOR, "spacer"))

    # End of the last real (non-zero-length) "kozak" domain surviving in the fused
    # frame, over EITHER hairpin -- whichever half actually keeps one after fuse().
    # Falls back to len(_fused) (no clipping at all) if, for some topology, neither
    # half has one, rather than guessing a cutoff.
    _kozak_ends_and = [
        _e for _kd in (_kept5p, _kept3p)
        for _n, (_s, _e) in _kd.items() if _n == "kozak" and _e > _s
    ]
    _visible_len_and = max(_kozak_ends_and) if _kozak_ends_and else len(_fused)

    _mfe_alone = folder.mfe(_fused)
    _matrix_alone = folder.base_pair_probabilities(_fused)
    _alone_structure = _mfe_alone.structure.replace("&", "")
    _r["structure_alone_svg"] = structure_svg(
        folder.layout_coordinates(_alone_structure), _fused, _alone_structure,
        f"{_r['d5']['rank']}+{_r['d3']['rank']} [{_r['spacer_label']}] — alone (OFF)",
        spans=_spans2, domain_fill=_domain_fill_and, visible_len=_visible_len_and,
        construct_len=len(_fused),
    )
    # No strand present besides the fused construct itself -- no junction to mark.
    _r["matrix_alone_png"] = bpp_heatmap_png(_fused, _matrix_alone, "alone: base-pair probabilities",
                                         figsize=_AND_BPP_FIGSIZE)

    # One crosshair per real strand junction present in THIS state's matrix: +A/+B have
    # exactly one (switch<->that trigger, in its own colour); +A+B has two, in strand order
    # (see both_triggers_extra): switch<->triggerB (pink) first, then triggerB<->triggerA
    # (blue). Only the +A+B state gets a structure plot (the card shows alone and +A+B
    # structures); all three get a matrix.
    _STATE_BOUNDARY_COLORS = {
        "+A": (_COLOR_A,),
        "+B": (_COLOR_B,),
        "+A+B": (_COLOR_B, _COLOR_A),
    }
    for _state_name, _extra in (("+A", (_r["d5"]["trigger"],)), ("+B", (_r["d3"]["trigger"],)),
                                 ("+A+B", both_triggers_extra(_r["d5"]["trigger"], _r["d3"]["trigger"]))):
        _strands = "&".join([_fused, *_extra])
        _concat = _fused + "".join(_extra)
        _matrix = folder.base_pair_probabilities(_strands)
        if _state_name == "+A+B":
            _mfe = folder.mfe(_strands)
            _both_structure = _mfe.structure.replace("&", "")
            _r["structure_+A+B_svg"] = structure_svg(
                folder.layout_coordinates(_both_structure), _concat, _both_structure,
                f"{_r['d5']['rank']}+{_r['d3']['rank']} — {_state_name}",
                spans=_spans2, domain_fill=_domain_fill_and, visible_len=_visible_len_and,
                construct_len=len(_fused),
            )
        _boundary_positions = []
        _pos = len(_fused)
        for _strand, _color in zip(_extra, _STATE_BOUNDARY_COLORS[_state_name], strict=True):
            _boundary_positions.append((_pos, _color))
            _pos += len(_strand)
        _r[f"matrix_{_state_name}_png"] = bpp_heatmap_png(
            _concat, _matrix, f"{_state_name}: base-pair probabilities",
            boundaries=_boundary_positions, figsize=_AND_BPP_FIGSIZE,
        )

    # Domain diagram: both hairpins in one fused-frame bar, doubled palette (A-tinted vs
    # B-tinted); built from _kept5p/_kept3p (payload excluded).
    _segments = []
    for _half_domains, _palette in ((_kept5p, _DOMAIN_COLOR_A), (_kept3p, _DOMAIN_COLOR_B)):
        for _name, (_s, _e) in _half_domains.items():
            if _e > _s:
                _segments.append((_s, _name, _e - _s, _palette.get(_name, "#B7C4BC"), _fused[_s:_e]))
    if _built.get("spacer_span"):
        _s0, _s1 = _built["spacer_span"]
        if _s1 > _s0:
            _segments.append((_s0, "spacer", _s1 - _s0, _SPACER_COLOR, _fused[_s0:_s1]))
    # Total fused construct length on the bar itself (triggers are never part of `_fused`).
    _r["layout_png"] = layout_bar_png(
        _segments, len(_fused),
        f"{_r['d5']['rank']}+{_r['d3']['rank']} [{_r['spacer_label']}] — domain layout "
        f"(both hairpins, fused length {len(_fused)} nt)",
    )
    _r["domain_rows"] = [(name, length, seq) for (_s, name, length, _c, seq) in sorted(_segments, key=lambda s: s[0])]

print(f"images built for {len(and_report_rows)} AND candidates")


In [ ]:
def render_and_report_html(rows):
    summary_trs = []
    for i, r in enumerate(rows, start=1):
        _built = r["built"]
        _su5_len = _built["domains_5p"]["stem_up"][1] - _built["domains_5p"]["stem_up"][0]
        _su3_len = _built["domains_3p"]["stem_up"][1] - _built["domains_3p"]["stem_up"][0]
        _fp5_len = _built["footprint_5p"][1] - _built["footprint_5p"][0]
        _fp3_len = _built["footprint_3p"][1] - _built["footprint_3p"][0]
        # ON/OFF cells are TWO labelled lines each (the column header explains both):
        # line 1 = the headline (ON: exact P(duplexes) of the +A+B state; OFF: the worst
        # illegitimate state's badness), line 2 = the exact per-side nt-counts of that SAME
        # state (never a number borrowed from a different state than the one OFF came from).
        _on_cell = (f'{_pct(r["on"])}'
                    f'<small class="scr-line">A {_nt(r["both"]["mean_trigger_bound_a_bpp"], _fp5_len)}'
                    f' &middot; B {_nt(r["both"]["mean_trigger_bound_b_bpp"], _fp3_len)}</small>')
        _off_cell = (f'{_pct(r["off"])} <small class="scr-line">{html.escape(str(r["off_source"]))} '
                     f'&middot; {html.escape(_off_term_label(r["off_criterion"] or "--")[0])}</small>'
                     f'<small class="scr-line">A {_nt(r["off_mean_stem_closed_a"], _su5_len)}'
                     f' &middot; B {_nt(r["off_mean_stem_closed_b"], _su3_len)}</small>')
        summary_trs.append(
            "<tr>"
            f"<td>{i}</td><td>{html.escape(r['spacer_label'])}</td>"
            f"<td>{html.escape(str(r['d5']['rank']))}+{html.escape(str(r['d3']['rank']))}</td>"
            f'<td class="num">{_on_cell}</td>'
            f'<td class="num">{_off_cell}</td>'
            f'<td class="num">{_ratio_text(r)}</td>'
            f'<td class="num">{_pct(r["on_minus_off"])}</td>'
            f'<td class="num">{_num(r["dg"]["d_ee_worst_alt"])}</td>'
            f'<td><a href="#and-{i}">detail &darr;</a></td>'
            "</tr>"
        )

    cards = []
    for i, r in enumerate(rows, start=1):
        _built = r["built"]
        _fp5_len = _built["footprint_5p"][1] - _built["footprint_5p"][0]
        _fp3_len = _built["footprint_3p"][1] - _built["footprint_3p"][0]
        _su5_len = _built["domains_5p"]["stem_up"][1] - _built["domains_5p"]["stem_up"][0]
        _su3_len = _built["domains_3p"]["stem_up"][1] - _built["domains_3p"]["stem_up"][0]
        # Card head: one labelled item per line (_metric_lines, cell 7); all exact.
        _off_terms = r.get("off_terms") or {}
        _terms_html = " &middot; ".join(
            ("<b>" if k == r["off_criterion"] else "")
            + f'{html.escape(_off_term_label(k)[0])} ({_off_term_label(k)[1]}): {_pct(v)}'
            + ("</b>" if k == r["off_criterion"] else "")
            for k, v in _off_terms.items()
        ) or "--"
        _head_lines = _metric_lines([
            ("ON &mdash; P(duplexes), all designed duplexes at once (exact)", _pct(r["on"])),
            ("ON &mdash; footprint bound (exact)",
             f'A {_nt(r["both"]["mean_trigger_bound_a_bpp"], _fp5_len)} &middot; '
             f'B {_nt(r["both"]["mean_trigger_bound_b_bpp"], _fp3_len)}'),
            (f'OFF &mdash; worst state: {html.escape(str(r["off_source"]))}',
             f'{_pct(r["off"])} (set by {html.escape(_off_term_label(r["off_criterion"] or "--")[0])}, exact)'),
            ("&nbsp;&nbsp;terms in that state", _terms_html),
            ("OFF &mdash; stem still closed in that state (exact)",
             f'A {_nt(r["off_mean_stem_closed_a"], _su5_len)} &middot; '
             f'B {_nt(r["off_mean_stem_closed_b"], _su3_len)}'),
            ("FAKES &mdash; stem still closed with a fake trigger present (exact)",
             f'A {_nt_pct(r["fakeA_mean_stem_closed"], _su5_len)} &middot; '
             f'B {_nt_pct(r["fakeB_mean_stem_closed"], _su3_len)}'),
            ("Ratio", _ratio_text(r)),
        ])

        # "How ON and OFF were computed": every OFF state's terms and the ON pair/cost numbers.
        def _pooled(results):
            return {f: _mean_or_none(results, f) for f in (
                "mean_stem_closed_a_bpp", "mean_stem_closed_b_bpp")}

        _and_states = []
        for _key, _label, _state, _mask in (
            ("alone", "alone", r["alone"], None), ("+A", "+A only", r["state_a"], "a"),
            ("+B", "+B only", r["state_b"], "b"),
            ("+fakeA", "+fakeA (mean)", _pooled(r["fakeA_results"]) if r["fakeA_results"] else None, None),
            ("+fakeB", "+fakeB (mean)", _pooled(r["fakeB_results"]) if r["fakeB_results"] else None, None),
            ("+A+fakeA", "+A+fakeA (mean)",
             _pooled(r["a_fakeA_results"]) if r["a_fakeA_results"] else None, "a"),
            ("+B+fakeB", "+B+fakeB (mean)",
             _pooled(r["b_fakeB_results"]) if r["b_fakeB_results"] else None, "b"),
        ):
            if _state is None:
                continue
            _t = _and_off_term(_key, _state, mask_side=_mask)
            _and_states.append((_key, _label, _t[5] or None, _t[1]))
        _numbers_html = (
            "<p class=\"scr-p\"><b>How ON and OFF were computed.</b> ON = P(duplexes) of the +A+B "
            "state = exp(&minus;(G_forced &minus; G)/RT), from two partition functions.</p>"
            + _on_numbers_html(
                r["on"], _designed_pair_counts_and(r),
                [("A", r["both"]["mean_trigger_bound_a_bpp"], _fp5_len),
                 ("B", r["both"]["mean_trigger_bound_b_bpp"], _fp3_len)], folder.rt)
            + "<p class=\"scr-p\">OFF = the largest state leak below (bold = the state that sets "
              f"OFF = {_pct(r['off'])}); a hairpin whose own trigger is present in a state is "
              "exempt there.</p>"
            + _off_numbers_html(_and_states, r["off_source"]))

        def _state_row(label, state, present, _fp5_len=_fp5_len, _fp3_len=_fp3_len,
                       _su5_len=_su5_len, _su3_len=_su3_len):
            # Per hairpin: trigger_bound / stem_closed / misfolded are exact partition-function
            # means (one number + nt-count). P(duplexes) is the exact joint probability that
            # all designed duplexes of the triggers present are formed; it exists only for the
            # +A (hairpin A's duplex, state["p_own_duplex"]), +B (hairpin B's) and +A+B
            # (both, state["p_on"]) states, None elsewhere -> "--". `present` = which REAL triggers this
            # state holds (see _plan_class, cell 7); every measured cell is shaded through it.
            def _c(metric, hairpin, value, inner):
                _cls = _plan_class(metric, hairpin, present, value)
                return f'<td class="num{" " + _cls if _cls else ""}">{inner}</td>'

            _p_on = state.get("p_on")
            if _p_on is None:
                _p_on = state.get("p_own_duplex")
            return (f"<tr><td>{label}</td>"
                    + _c("bound", "A", state["mean_trigger_bound_a_bpp"],
                         _pct_exact(state["mean_trigger_bound_a_bpp"], _fp5_len))
                    + _c("closed", "A", state["mean_stem_closed_a_bpp"],
                         _pct_exact(state["mean_stem_closed_a_bpp"], _su5_len))
                    + _c("misfolded", "A", state["mean_misfolded_a_bpp"],
                         _pct_exact(state["mean_misfolded_a_bpp"], _fp5_len))
                    + _c("bound", "B", state["mean_trigger_bound_b_bpp"],
                         _pct_exact(state["mean_trigger_bound_b_bpp"], _fp3_len))
                    + _c("closed", "B", state["mean_stem_closed_b_bpp"],
                         _pct_exact(state["mean_stem_closed_b_bpp"], _su3_len))
                    + _c("misfolded", "B", state["mean_misfolded_b_bpp"],
                         _pct_exact(state["mean_misfolded_b_bpp"], _fp3_len))
                    + _c("joint", None, _p_on, _pct(_p_on))
                    + "</tr>")

        decomp_rows = (
            _state_row("alone (no trigger)", r["alone"], ())
            + _state_row("+A only", r["state_a"], ("A",))
            + _state_row("+B only", r["state_b"], ("B",))
            + _state_row("<b>+A+B (ON)</b>", r["both"], ("A", "B"))
        )
        # Fake rows: the mean of the per-draw exact means (None if any draw lacks a value).
        _pool_fields = ("mean_trigger_bound_a_bpp", "mean_stem_closed_a_bpp", "mean_misfolded_a_bpp",
                        "mean_trigger_bound_b_bpp", "mean_stem_closed_b_bpp", "mean_misfolded_b_bpp")
        if r["fakeA_results"]:
            _mean_fa = {k: _mean_or_none(r["fakeA_results"], k) for k in _pool_fields}
            decomp_rows += _state_row(f"+fakeA (mean of {len(r['fakes_a'])})", _mean_fa, ())
        if r["fakeB_results"]:
            _mean_fb = {k: _mean_or_none(r["fakeB_results"], k) for k in _pool_fields}
            decomp_rows += _state_row(f"+fakeB (mean of {len(r['fakes_b'])})", _mean_fb, ())

        energy_rows = f"""
          <tr><td>alone</td><td class="num">{_num(r["dg"]["ee_off"])}</td></tr>
          <tr><td>trigger A alone</td><td class="num">{_num(r["ee_trigger_a"])}</td></tr>
          <tr><td>trigger B alone</td><td class="num">{_num(r["ee_trigger_b"])}</td></tr>
          <tr><td>+A</td><td class="num">{_num(r["dg"]["ee_t5"])}</td></tr>
          <tr><td>+B</td><td class="num">{_num(r["dg"]["ee_t3"])}</td></tr>
          <tr><td>+A+B</td><td class="num">{_num(r["dg"]["ee_both"])}</td></tr>
          <tr><td><b>&Delta;(ON) = ee(+A+B) &minus; min(alone, +A, +B)</b></td>
              <td class="num">{_num(r["dg"]["d_ee_worst_alt"])}</td></tr>
        """

        # The switch sequence first, in one piece (5' -> 3') and only through the Kozak (the
        # reporter-gene payload is folded but not shown), then the same sequence cut into its
        # domains below.
        _kozak_end = max(e for n, (s, e) in r["built"]["domains_3p"].items() if n == "kozak")
        _full_seq = r["built"]["fused"][:_kozak_end]
        domain_block = (
            f'<div class="scr-seq-block"><span class="label">Switch sequence through the Kozak ({len(_full_seq)} nt, 5\' to 3\')</span>'
            f'<span class="scr-mono" style="word-break:break-all;">{html.escape(_full_seq)}</span></div>'
        ) + "".join(
            f'<div class="scr-seq-block"><span class="label">{html.escape(name)} ({length} nt)</span>'
            f'<span class="scr-mono">{html.escape(seq)}</span></div>'
            for name, length, seq in r["domain_rows"]
        )
        # The spacer's own length, or an explicit "no spacer" (a real, deliberate arm of
        # SPACER_VARIANTS, not a missing measurement). Own line under Trigger B.
        if r["spacer_seq"]:
            _spacer_note = (f'<br><b>Spacer ({len(r["spacer_seq"])} nt):</b> '
                            f'<span class="scr-mono">{html.escape(r["spacer_seq"])}</span>'
                            f' (worst complementary run {r["spacer_worst"]} nt)')
        else:
            _spacer_note = "<br><b>Spacer:</b> none (no-spacer variant)"

        cards.append(f"""
        <div class="scr-candidate" id="and-{i}">
          <div class="scr-candidate-head">
            <h3>#{i} — {html.escape(str(r["d5"]["rank"]))}+{html.escape(str(r["d3"]["rank"]))}
              [{html.escape(r["spacer_label"])}]</h3>
          </div>
          {_head_lines}
          <p class="scr-p"><b>Trigger A ({len(r["d5"]["trigger"])} nt):</b> <span class="scr-mono">{html.escape(r["d5"]["trigger"])}</span><br>
             <b>Trigger B ({len(r["d3"]["trigger"])} nt):</b> <span class="scr-mono">{html.escape(r["d3"]["trigger"])}</span>{_spacer_note}</p>
          {_numbers_html}
          <p class="scr-p">Energies are ensemble &Delta;G in kcal/mol; more negative = more stable.</p>
          <div class="scr-table-wrap scr-table-energy" style="max-width:420px;"><table>
            <thead><tr><th>energy</th><th class="num">kcal/mol</th></tr></thead>
            <tbody>{energy_rows}</tbody>
          </table></div>
          <p class="scr-p">Per state and hairpin: bound / closed / misfolded are exact ensemble means.
            P(duplexes) is the exact probability that all designed footprint-trigger duplexes of the
            triggers present are formed in the same structure: hairpin A's duplex in the +A row,
            hairpin B's in the +B row, both at once in the +A+B row (not the product of the two
            bound columns). It is <code>--</code> in the states with no real trigger.</p>
          <div class="scr-table-wrap scr-table-and-decomp"><table style="table-layout:fixed; width:100%;">
            <colgroup>
              <col style="width:16%">
              <col style="width:11%"><col style="width:12%"><col style="width:11%">
              <col style="width:11%"><col style="width:12%"><col style="width:11%">
              <col style="width:16%">
            </colgroup>
            <thead><tr><th>state</th>
              <th class="num" colspan="3">hairpin A <small>bound / closed / misfolded -- exact</small></th>
              <th class="num" colspan="3">hairpin B <small>bound / closed / misfolded -- exact</small></th>
              <th class="num">P(duplexes)<br><small>exact &middot; own duplex(es) formed</small></th></tr></thead>
            <tbody>{decomp_rows}</tbody>
          </table></div>
          <p class="scr-note">{_plan_legend_html()}</p>
          <img class="scr-img" src="data:image/png;base64,{r["layout_png"]}">
          <div class="scr-seq-block">{domain_block}</div>
          <div class="scr-structure-stack">
            <figure style="margin:0 0 10px 0;break-inside:avoid;page-break-inside:avoid;">{r["structure_alone_svg"]}<figcaption>alone (OFF)</figcaption></figure>
            <figure style="margin:0 0 10px 0;break-inside:avoid;page-break-inside:avoid;">{r["structure_+A+B_svg"]}
              <figcaption>+A+B (ON), triggers hidden</figcaption></figure>
          </div>
          <div class="scr-structure-grid scr-grid-2">
            <figure><img src="data:image/png;base64,{r["matrix_alone_png"]}"><figcaption>alone: probability matrix</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+A_png"]}"><figcaption>+A: probability matrix (dashed crosshair = A boundary)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+B_png"]}"><figcaption>+B: probability matrix (dashed crosshair = B boundary)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+A+B_png"]}"><figcaption>+A+B: probability matrix (two crosshairs: B strand first, pink, then A strand, blue)</figcaption></figure>
          </div>
          <p class="scr-note">Structure plots are clipped at kozak (payload dropped). A dangling
            ring marks a real base pair whose partner is not drawn: amber = into this construct's
            own hidden payload tail, red = to a hidden trigger strand.</p>
        </div>
        """)

    _off_states = "alone, +A, +B, mean +fakeA, mean +fakeB, mean +A+fakeA, mean +B+fakeB"
    _spacer_variants = ", ".join(SPACER_VARIANTS)

    return f"""<!DOCTYPE html><html><head><meta charset="utf-8">
    <title>AND ON/OFF report</title><style>{_REPORT_CSS}</style></head>
    <body class="scr"><div class="scr-wrap">
      <p class="scr-eyebrow">CERNAL · eukaryotic toehold switch · AND (two inputs)</p>
      <h1>AND ON/OFF report</h1>
      <div class="scr-dek"><ul class="scr-intro">
        <li><b>Source:</b> <span class="scr-mono">{html.escape(CSV_PATH)}</span>.</li>
        <li><b>Top N singles (pool):</b> the top {AND_GRID_N} singles by single-switch ON/OFF
          ratio are fused into every ordered pair (5' half cut before its own
          <code>{html.escape(CUT_5P_BEFORE)}</code>, 3' half kept after its own
          <code>{html.escape(KEEP_3P_AFTER)}</code>); spacer variants: {html.escape(_spacer_variants)},
          each reported separately. The top {AND_REPORT_N} pairs by ON&minus;OFF are shown (the ratio is shown too, not used for the order).</li>
        <li><b>Method:</b> every value is an exact partition-function quantity at
          {TEMPERATURE_C:.1f}&deg;C; nothing is sampled. The green/red cell shading uses
          OPEN_THRESHOLD ({OPEN_THRESHOLD:.0%}) only as a HIGH/LOW cutoff.</li>
        <li><b>ON:</b> P(duplexes | +A+B), the exact probability that both designed
          footprint-trigger duplexes are present in the same structure.</li>
        <li><b>OFF:</b> the worst stem opening over the states {{{_off_states}}} (see the Glossary).
          Fakes: up to {AND_N_FAKES} per side; a fake shares at most {FAKE_MAX_SEQ_OVERLAP_NT} nt
          of identical sequence with either real trigger of the pair.</li>
        <li><b>Ratio:</b> ON% / max(OFF%, {OFF_FLOOR_PCT:g}%), in percentage-point units; an OFF below {OFF_FLOOR_PCT:g}% is replaced by {OFF_FLOOR_PCT:g}% (row marked *).</li>
        <li><b>Trigger overlap screen:</b> a pair whose two real triggers share a complementary
          run or a same-direction identity run longer than {FAKE_MAX_SEQ_OVERLAP_NT} nt is
          excluded before scoring.</li>
      </ul></div>
      <section>
        <h2>Glossary</h2>
        <div class="scr-table-wrap"><table>
          <tbody>
            <tr><td style="width:150px"><b>P(duplexes)</b> in +A / +B</td><td>The same
              exact quantity as ON, for one hairpin: the probability that every designed pair of
              that hairpin's own footprint with its trigger is present at once (hairpin A in
              the +A row, hairpin B in the +B row), the full designed alignment. The bound column
              next to it is only the average over the footprint; this is the probability that
              the complete duplex is formed.</td></tr>
            <tr><td style="width:150px"><b>ON</b> / <b>P(duplexes)</b></td><td>Exact probability that
              ALL designed footprint-trigger base pairs of the +A+B state are present in the same
              structure of the Boltzmann ensemble
              (<code>FoldEngine.constrained_probability</code>: <code>exp(-(G_forced - G)/RT)</code>
              from two partition functions). The designed pairs are the pairs between each
              footprint and its trigger by sequence design (every complementary position of the trigger aligned
              antiparallel from the footprint start; a designed mismatch is skipped); the event is strict --
              every one of those pairs must be present at once, so it is lower than the probability of
              any single one of them. It is <code>--</code> (not zero) when a duplex has no designed
              pair. The two trigger
              strands are appended trigger B first, then trigger A
              (<code>both_triggers_extra</code>), because the folding model only represents
              structures nested along the written strand order.</td></tr>
            <tr><td><b>footprint bound</b></td><td>Exact expected number of nt of a
              hairpin's footprint (toehold + stem_up) paired to an external strand, shown as
              nt out of the footprint length. Per hairpin (A, B).</td></tr>
            <tr><td><b>OFF</b></td><td>The worst stem opening over the illegitimate states {{{_off_states}}}; not 1 &minus; ON. A state's leak is the largest <i>stem-loosening A/B</i> (1 &minus; stem_closed: the share of stem_up not paired to its own stem_down, whatever the cause), taken for each hairpin whose own real trigger is absent in that state (a hairpin whose own trigger is present is exempt). So +A is scored on
              hairpin B only, +B on hairpin A only, +A+fakeA on B only, +B+fakeB on A only; alone,
              +fakeA and +fakeB on both. Each card lists the terms of its worst state.</td></tr>
            <tr><td><b>stem still closed</b></td><td>stem_closed of a hairpin in
              the worst OFF state, as nt out of its stem_up length (higher is better).</td></tr>
            <tr><td><b>ratio</b></td><td>ON% / max(OFF%, {OFF_FLOOR_PCT:g}%), in percentage-point units (an OFF below {OFF_FLOOR_PCT:g}% is replaced by it; row marked *); higher is
              better, and OFF = 0 still gives a finite value.</td></tr>
            <tr><td><b>dG / &Delta;(ON)</b></td><td>ee(+A+B) minus the most negative of ee(alone),
              ee(+A), ee(+B), in kcal/mol (ensemble &Delta;G); more negative means the fully
              bound state is more stable than every partially bound one.</td></tr>
            <tr><td><b>trigger overlap screen</b></td><td>A pair's two real triggers are compared
              with each other before the pair is fused: longest complementary run (wobble
              included) and longest same-direction identity run. A pair exceeding
              {FAKE_MAX_SEQ_OVERLAP_NT} nt on either check is dropped and not scored.</td></tr>
            <tr><td><b>FAKES (in each card)</b></td><td>fakeA is a trigger not
              meant for hairpin A, fakeB the same for hairpin B (each sharing at most
              {FAKE_MAX_SEQ_OVERLAP_NT} nt of identical sequence with either real trigger). Value: mean over that side's fake draws of stem_closed of the
              side's own hairpin, as nt out of its stem_up length with the same mean as a
              percentage. Higher is better; 100% means the stem stays fully closed with the fake
              present. "--" means no qualifying fake was found for that side.</td></tr>
            <tr><td><b>trigger_bound A/B</b></td><td>Expected fraction of the
              hairpin's footprint paired to an external strand (nt out of footprint length).</td></tr>
            <tr><td><b>stem_closed A/B</b></td><td>Expected fraction of the hairpin's
              stem_up paired to its own stem_down (nt out of stem_up length): the hairpin's
              intended OFF structure being closed.</td></tr>
            <tr><td><b>misfolded A/B</b></td><td>Expected fraction of the hairpin's
              footprint paired to neither an external strand nor its own stem_down (nt out of
              footprint length). A structural risk signal, not a measured expression leak.</td></tr>
          </tbody>
        </table></div>
      </section>
      <section>
        <h2>Summary</h2>
        <p class="scr-p"><b>How to read ON and OFF.</b>
          <b>ON&nbsp;%</b> is P(duplexes): the probability, over the whole Boltzmann ensemble, that
          every designed footprint-trigger pair is formed at the same time (the share of molecules
          fully switched on). The small line under it is the average number of footprint
          nucleotides bound, out of the footprint length.
          <b>OFF&nbsp;%</b> is the leak in the worst OFF state, a state where the hairpin's own
          trigger is absent (alone, a fake trigger, only one of the two real triggers, or one real plus a fake). It is the share of stem_up that is open, i.e. NOT paired to its own stem_down (stem loosening = 1 &minus; stem closed), whatever the cause: a fake trigger, the other trigger, or the hairpin opening on its own. It is an
          average share of nucleotides, not the probability of one event, so ON and OFF are not the
          same kind of number. The small lines under it name the state and the hairpin, and give the
          stem nucleotides still closed there. <b>ratio</b> = ON&nbsp;% / max(OFF&nbsp;%,
          {OFF_FLOOR_PCT:g}&nbsp;%).</p>
        <div class="scr-table-wrap scr-table-summary"><table>
          <thead><tr><th>#</th><th>spacer</th><th>pair</th>
            <th class="num">ON<small class="scr-sub">P(all designed pairs formed, A and B)<br>avg footprint nt bound, per hairpin</small></th>
            <th class="num">OFF<small class="scr-sub">leak in worst OFF state<br>state &middot; term<br>stem nt still closed there, per hairpin</small></th>
            <th class="num">ratio</th>
            <th class="num">ON&minus;OFF</th><th class="num">&Delta;(ON)</th>
            <th></th></tr></thead>
          <tbody>{"".join(summary_trs)}</tbody>
        </table></div>
        {_ratio_floor_note(rows, OFF_FLOOR_PCT)}
        <p class="scr-p">A "--" cell means the value was not measured, not zero. The fake-trigger
          check (FAKES) is in each candidate's card.</p>
      </section>
      <section><h2>Candidates</h2>{"".join(cards)}</section>
      <footer class="scr-footer">Generated {datetime.datetime.now().isoformat(timespec="seconds")}
        · OPEN_THRESHOLD={OPEN_THRESHOLD} (plan colouring only) · AND_N_FAKES={AND_N_FAKES}/side
        · all values exact (partition function); nothing sampled
        · FoldEngine {folder.versions()}</footer>
    </div></body></html>"""


_and_html = render_and_report_html(and_report_rows)
_ts2 = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
and_report_html_path = _report_root / f"and_report_{_ts2}.html"
and_report_html_path.write_text(_and_html)
print(f"wrote {and_report_html_path.stat().st_size:,} bytes -> {and_report_html_path}")

and_report_pdf_path = _render_pdf(and_report_html_path)


## Snapshot export (the matching load is in cell 9)

In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # SNAPSHOT EXPORT -- timestamped JSON holding exactly what the report cells need:
    # single_report_rows and and_report_rows minus images (the base64 PNGs rebuild in
    # seconds from the sequences and would bloat the file). Restored by cell 9.
    # ==============================================================================
    def _strip_images(rows):
        return [{k: v for k, v in r.items() if not k.endswith("_png")} for r in rows]

    _snapshot = {
        "exported_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "csv_path": CSV_PATH,
        "temperature_c": TEMPERATURE_C,
        "open_threshold": OPEN_THRESHOLD,
        "single_report_rows": _strip_images(single_report_rows),
        "and_report_rows": _strip_images(and_report_rows),
        "single_scored_pool": single_scored_pool,
    }
    _snap_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
    _snapshot_path = _report_root / f"and_eu_report_snapshot_{_snap_ts}.json"
    _snapshot_path.write_text(json.dumps(_snapshot, default=str))
    print(f"wrote snapshot -> {_snapshot_path} ({_snapshot_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping snapshot export.")


In [ ]:
print("REPORTS ready:")
print(f"  single-switch HTML: {single_report_html_path}")
print(f"  single-switch PDF:  {single_report_pdf_path}")
print(f"  AND HTML:           {and_report_html_path}")
print(f"  AND PDF:            {and_report_pdf_path}")
print(f"  {len(single_report_rows)} single-switch candidates, {len(and_report_rows)} AND candidates, "
      f"both ranked by ON - OFF.")


# Verification

Independent sanity checks on this notebook's own output, kept in the file (not a
standalone script -- a script copy of notebook code once produced a silent indexing bug on
this project). Each check either re-derives a number a different way or asserts an
invariant the pipeline promises but does not itself check.


In [ ]:
# 1) Both report lists are sorted by ON - OFF, descending (None last) -- the summary table
# and the candidate-card order must agree. Single rows: ON - OFF rounded to 1 decimal in
# percentage points, ties by dG ascending (`_single_sort_key`, cell 7); AND rows: ON - OFF
# (the ratio is displayed but orders neither list).
_keys = [_single_sort_key(r["on_minus_off"], r["d_ee"]) for r in single_report_rows]
assert _keys == sorted(_keys), "single_report_rows is not sorted by (rounded ON-OFF desc, dG asc)"
_and_keys = [(r["on_minus_off"] is None, 0.0 if r["on_minus_off"] is None else -r["on_minus_off"])
             for r in and_report_rows]
assert _and_keys == sorted(_and_keys), "and_report_rows is not sorted by ON-OFF descending (None last)"
print(f"ranking check OK: single ({len(single_report_rows)} rows) and AND "
      f"({len(and_report_rows)} rows) are both sorted by ON - OFF, descending")


In [ ]:
# 2) Trap 1, re-checked directly: with NO trigger/fake present, trigger_bound must be
# essentially 0 for every report row's "alone" state -- nothing external is there to bind
# the footprint, so a nonzero reading would mean trigger_bound is counting intramolecular
# pairing as external.
_bad = [r["rank"] for r in single_report_rows if r["alone"]["mean_trigger_bound_bpp"] > 0.02]
assert not _bad, f"single-switch alone-state trigger_bound > 2% for ranks {_bad} -- Trap 1 regression"
_bad_and = [
    (r["d5"]["rank"], r["d3"]["rank"])
    for r in and_report_rows
    if r["alone"]["mean_trigger_bound_a_bpp"] > 0.02 or r["alone"]["mean_trigger_bound_b_bpp"] > 0.02
]
assert not _bad_and, f"AND alone-state trigger_bound > 2% for pairs {_bad_and} -- Trap 1 regression"
print(f"Trap 1 check OK: alone-state trigger_bound ~0 for all {len(single_report_rows)} single "
      f"+ {len(and_report_rows)} AND report rows")


In [ ]:
# 3) The three-way decomposition partitions each footprint's possible partners without
# double-counting: for ANY footprint position, the probability of being unpaired, paired
# to its own stem_down (stem_closed's territory), paired externally (trigger_bound's) or
# paired elsewhere intramolecularly (misfolded's) are four mutually exclusive outcomes
# that must add up to exactly 1. This re-derives that from the raw base-pair-probability
# matrix of one real report row's ON state (an independent sum over matrix entries, not a
# call into _joint_state_parallel), and checks that the module's trigger_bound and
# misfolded means equal the means of those per-position sums.
_check_row = single_report_rows[0]
_d = _check_row["design"]
_switch, _trigger = _d["switch"], _d["trigger"]
_toehold, _stem_up, _stem_down = _d["domains"]["toehold"], _d["domains"]["stem_up"], _d["domains"]["stem_down"]
_footprint = (_toehold[0], _stem_up[1])
_n_self = len(_switch)
_f0, _f1 = _footprint
_sd0, _sd1 = _stem_down

_m = folder.base_pair_probabilities(f"{_switch}&{_trigger}")
_ext_sum = _mis_sum = 0.0
for _i in range(_f0, _f1):
    _row_p = _m[_i]
    _p_ext = sum(_row_p[_j] for _j in range(_n_self, len(_row_p)))
    _p_own = sum(_row_p[_j] for _j in range(_sd0, _sd1))
    _p_mis = sum(_row_p[_j] for _j in range(_n_self) if not (_sd0 <= _j < _sd1))
    _p_unp = 1.0 - sum(_row_p)
    assert _p_unp > -1e-6, f"position {_i}: pairing probabilities add up to {1 - _p_unp:.6f} > 1"
    assert abs((_p_unp + _p_ext + _p_own + _p_mis) - 1.0) < 1e-6
    _ext_sum += _p_ext
    _mis_sum += _p_mis
_n_fp = _f1 - _f0
assert abs(_ext_sum / _n_fp - _check_row["real"]["mean_trigger_bound_bpp"]) < 1e-9
assert abs(_mis_sum / _n_fp - _check_row["real"]["mean_misfolded_bpp"]) < 1e-9
print(f"three-way partition check OK (exact): rank {_check_row['rank']}'s ON state, every footprint "
      f"position's unpaired + external + own-stem_down + other-intramolecular probabilities sum to 1; "
      f"module means match ({_ext_sum / _n_fp:.3f} external, {_mis_sum / _n_fp:.3f} misfolded)")


In [ ]:
# 4) Fake rule, re-checked directly on every fake actually used in the report rows: its
# longest same-direction identical run against each real trigger it must not resemble
# (single: the candidate's trigger; AND: BOTH real triggers of the pair) is at most
# FAKE_MAX_SEQ_OVERLAP_NT. Hard rule, so asserted.
from _joint_state_parallel import _longest_identity_run

_max_run, _n_checked = 0, 0
for _r in single_report_rows:
    _reals = (_r["design"]["trigger"],)
    for _wd in _r["fakes"]:
        for _real in _reals:
            _run = _longest_identity_run(_real, _wd["trigger"])
            assert _run <= FAKE_MAX_SEQ_OVERLAP_NT, (_r["rank"], _wd["rank"], _run)
            _max_run = max(_max_run, _run)
        _n_checked += 1
for _r in and_report_rows:
    _reals = (_r["d5"]["trigger"], _r["d3"]["trigger"])
    for _wd in _r["fakes_a"] + _r["fakes_b"]:
        for _real in _reals:
            _run = _longest_identity_run(_real, _wd["trigger"])
            assert _run <= FAKE_MAX_SEQ_OVERLAP_NT, (_r["key"], _wd["rank"], _run)
            _max_run = max(_max_run, _run)
        _n_checked += 1

print(f"fake identity check: {_n_checked} fakes checked, max identical run observed "
      f"{_max_run} nt (cap {FAKE_MAX_SEQ_OVERLAP_NT} nt)")

In [ ]:
# 5) Two-trigger strand order (both_triggers_extra). For the top real AND pair, on the
# notebook's ACTUAL +A+B strand list, BOTH triggers must bind their own footprint (mean
# pairing probability > 0.5). The reversed (A-then-B) order is folded only to document
# why the order matters: it crosses A<->tA with B<->tB (a pseudoknot ViennaRNA cannot
# represent), so trigger B's own footprint binding collapses.
_row = and_report_rows[0]
_fused = _row["built"]["fused"]
_ta, _tb = _row["d5"]["trigger"], _row["d3"]["trigger"]
_fpa, _fpb = tuple(_row["built"]["footprint_5p"]), tuple(_row["built"]["footprint_3p"])


def _mean_bound_to(matrix, footprint, partner_span):
    """Mean over footprint positions of the total pairing probability into partner_span."""
    return sum(sum(matrix[i][j] for j in range(*partner_span)) for i in range(*footprint)) / (
        footprint[1] - footprint[0]
    )


_n = len(_fused)
_extra = both_triggers_extra(_ta, _tb)  # (trigger_b, trigger_a)
assert _extra == (_tb, _ta)
_m_fwd = folder.base_pair_probabilities("&".join([_fused, *_extra]))
_span_b = (_n, _n + len(_tb))
_span_a = (_n + len(_tb), _n + len(_tb) + len(_ta))
_bind_a = _mean_bound_to(_m_fwd, _fpa, _span_a)
_bind_b = _mean_bound_to(_m_fwd, _fpb, _span_b)
assert _bind_a > 0.5 and _bind_b > 0.5, (
    f"notebook +A+B order: trigger A -> footprint A {_bind_a:.2f}, trigger B -> footprint B "
    f"{_bind_b:.2f}; both must exceed 0.5"
)

_m_rev = folder.base_pair_probabilities("&".join([_fused, _ta, _tb]))  # the WRONG order
_rev_a = _mean_bound_to(_m_rev, _fpa, (_n, _n + len(_ta)))
_rev_b = _mean_bound_to(_m_rev, _fpb, (_n + len(_ta), _n + len(_ta) + len(_tb)))
print(f"pair {_row['d5']['rank']}+{_row['d3']['rank']} [{_row['spacer_label']}] +A+B: "
      f"notebook order (B then A): A {_bind_a:.2f}, B {_bind_b:.2f} | "
      f"reversed (A then B): A {_rev_a:.2f}, B {_rev_b:.2f}  (informational: reversed order is expected to be lower for trigger B)")


In [ ]:
# 6) Joint ON probability sanity (exact): for the top single row and the top AND row,
# p_on must exist, lie in (0, 1], and not exceed the probability of any ONE of the designed
# pairs it requires (P(all pairs) <= min over pairs of P(that pair), read from the exact
# base-pair-probability matrix). Printed with the numbers; asserted with a small tolerance.
from _joint_state_parallel import both_triggers_extra, designed_pairs

_TOL = 1e-6


def _weakest_designed_pair(strands, pairs):
    matrix = folder.base_pair_probabilities(strands)
    return min(matrix[i][j] for i, j in pairs)


_row1 = single_report_rows[0]
_d1 = _row1["design"]
_p1 = _row1["real"]["p_on"]
assert _p1 is not None, f"single rank {_row1['rank']}: p_on is None"
assert 0.0 < _p1 <= 1.0, f"single rank {_row1['rank']}: p_on {_p1} outside (0, 1]"
_pairs1 = designed_pairs(
    _d1["switch"], ((_d1["domains"]["toehold"][0], _d1["trigger"], len(_d1["switch"])),))
_w1 = _weakest_designed_pair(f"{_d1['switch']}&{_d1['trigger']}", _pairs1)
assert _p1 <= _w1 + _TOL, f"single rank {_row1['rank']}: p_on {_p1:.4f} > weakest pair {_w1:.4f}"
print(f"single rank {_row1['rank']}: p_on {_p1:.4f} <= weakest designed pair {_w1:.4f} "
      f"({len(_pairs1)} pairs)")

_row2 = and_report_rows[0]
_p2 = _row2["both"]["p_on"]
assert _p2 is not None, f"AND {_row2['d5']['rank']}+{_row2['d3']['rank']}: p_on is None"
assert 0.0 < _p2 <= 1.0, f"AND p_on {_p2} outside (0, 1]"
_b2 = _row2["built"]
_ta2, _tb2 = _row2["d5"]["trigger"], _row2["d3"]["trigger"]
_extra2 = both_triggers_extra(_ta2, _tb2)
_offset = {s: len(_b2["fused"]) + sum(len(x) for x in _extra2[:k]) for k, s in enumerate(_extra2)}
_pairs2 = designed_pairs(_b2["fused"], (
    (_b2["footprint_5p"][0], _ta2, _offset[_ta2]), (_b2["footprint_3p"][0], _tb2, _offset[_tb2])))
_w2 = _weakest_designed_pair("&".join([_b2["fused"], *_extra2]), _pairs2)
assert _p2 <= _w2 + _TOL, f"AND p_on {_p2:.4f} > weakest designed pair {_w2:.4f}"
print(f"AND {_row2['d5']['rank']}+{_row2['d3']['rank']} [{_row2['spacer_label']}]: p_on {_p2:.4f} "
      f"<= weakest designed pair {_w2:.4f} ({len(_pairs2)} pairs)")
